In [ ]:
# copyright ################################# #
# This file is part of the wakis Package.     #
# Copyright (c) CERN, 2024.                   #
# ########################################### #

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eshipton/Wakis/blob/main/tutorials/07_light_sources/07-10_pcsel/pcsel.ipynb)

# Tutorial 7.10 · Photonic-crystal surface-emitting laser (PCSEL): band-edge modes and 3D coupled-wave theory

**Photonic-crystal surface-emitting lasers (PCSELs)** achieve watt-to-kilowatt class, diffraction-limited surface-normal lasing by operating at the two-dimensional (2D) $\Gamma$-point band edge ($\mathbf{G} = (\pm 2\pi/a, 0), (0, \pm 2\pi/a)$) of a square-lattice photonic crystal (PhC)slab integrated adjacent to a semiconductor quantum-well (QW) active layer [1–4]. Unlike vertical-cavity surface-emitting lasers (VCSELs), whose small cavity volume limits single-mode power to a few milliwatts, or edge-emitting distributed-feedback (DFB) lasers, whose elliptical facet beam suffers from filamentation at wide apertures, a PCSEL couples four counter-propagating and orthogonally propagating in-plane guided waves across a $100\;\mu\text{m}$ to $3\;\text{mm}$ aperture while diffracting coherent first-order radiation vertically along the surface normal ($\pm\hat{\mathbf{z}}$).

In this tutorial we combine full-wave 3D Finite Integration Technique (`wakis.SolverFIT3D`) Bloch unit-cell simulations with the Noda group's **three-dimensional coupled-wave theory (3D-CWT)** implemented in `wakis.pcsel_cwt` [1, 2] to analyze:
1. **3D FIT unit-cell band-edge modes A–D**: frequencies, symmetry-protected bound states in the continuum (BICs) vs radiative doublet $Q$ factors (`ResonanceMonitor` with `harminv`), $C_{4v}$ field patterns (`DFTMonitor`), and near-$\Gamma$ dispersion along $\Gamma\text{--}X$ and $\Gamma\text{--}M$.
2. **First-principles 3D-CWT coupling coefficients** ($\kappa_{1\text{D}}$, $\kappa_{2\text{D}}$, $R$, $I$, $\mu$) from the vertical fundamental $\text{TE}_0$ waveguide mode and 1D vertical Green's functions $\Phi_{\text{rad}}(z)$, $\Phi_{m,n}^{\text{guided}}(z, z')$, including both the transverse TE ($\xi_{m,n}$) higher-order channel [1] and an ad hoc local depolarisation term $\propto \xi\xi/\varepsilon_{\text{avg}}$ (not Liang *et al.*'s $\zeta_{m,n}$ form), benchmarked against the 3-level 3D FIT convergence ladder and across hole radii $r/a \in [0.08, 0.28]$ to identify where single-mode perturbative CWT breaks down.
3. **Finite-size $L\times L$ cavity scaling and the double-lattice PCSEL** [3, 4]: why a single-lattice PCSEL suffers multimode threshold-margin collapse **in absolute $\Delta\alpha_{\text{th}}$** ($\Delta\alpha_{\text{th}}(L) = \Delta c_v L^{-2} + \Delta c_\parallel L^{-3} \propto L^{-2}$ asymptotically, with measured local log-slopes $-1.79 \to -2.48 \to -2.36 \to -2.21 \to -2.12 \to -2.07$ across $L \in [50, 3000]\;\mu\text{m}$) **and in relative margin $\Delta\alpha_{\text{th}}/\alpha_{\text{th}}$ for the $\alpha_v$-matched synthetic-phase model** at large aperture $L$, and how introducing a second air hole shifted by $(\Delta x, \Delta y) = (0.2768a, 0.2768a)$ with $r_2/r_1 = 0.8882$ achieves one-directional $180^\circ$ coupling ($C_{12} = 0$, a non-Hermitian exceptional point where $R = 0$ and $|I| = \mu$), overtaking the $\alpha_v$-matched synthetic-phase single lattice above $L_{\text{cross}} = 149.4\;\mu\text{m}$ ($30.8\times\text{--}95.6\times$ larger relative margin across $L = 0.5\text{--}3\;\text{mm}$). By contrast, the unperturbed **circular single lattice** ($\alpha_{v,0} = 0$) keeps $\approx 11.1\%$ relative margin at every $L$ ($11.11\%$ at the $3\;\text{mm}$ sweep edge), so the double lattice ($8.93\%$ at $2\;\text{mm}$, $4.18\%$ at $3\;\text{mm}$) crosses back below the circular single lattice at a second crossover near $1.76\;\text{mm}$ ($1755.72\;\mu\text{m}$ by log interpolation), with relative-margin ratio $0.80\times$ at $2\;\text{mm}$ and **$0.38\times$** at $3\;\text{mm}$.

---

## Learning Goals

1. **Understand 2D $\Gamma$-point band-edge feedback in a square-lattice PCSEL**: Decompose the in-plane guided field into four basic waves $(R_x, S_x, R_y, S_y)$ at $\beta_0 = 2\pi/a$ and classify the four $\Gamma$-point eigenmodes **A** ($A_2$, antisymmetric dark BIC), **B** ($B_2$, quadrupolar dark BIC), and **C, D** ($E$ degenerate dipole doublet, surface-emitting) under the $C_{4v}$ point group.
2. **Derive the 3D coupled-wave theory (3D-CWT) from Maxwell's equations**: Compute the vertical fundamental $\text{TE}_0$ mode $\Theta_0(z)$, the radiative Green's function $\Phi_{\text{rad}}(z)$ (satisfying the optical theorem $-\operatorname{Im}(\mathcal{I}_{\text{rad}}) = k_B n_{\text{clad}}(|\Phi_{\text{rad}}(z_{\min})|^2 + |\Phi_{\text{rad}}(z_{\max})|^2)$), and the higher-order guided Green's functions $\Phi_{m,n}(z, z')$ in `wakis.pcsel_cwt`.
3. **Extract band-edge frequencies, radiative $Q$, and Bloch dispersion in 3D FIT**: Excited by $C_{4v}$-Projected `CurrentSource` dipole pairs in a vertically graded mesh ($\ge 22.7\;\text{cells}/\lambda$ in the $n = 3.52$ active core, $8.64\;\text{cells}$ across the hole diameter), use `ResonanceMonitor` (`harminv`) and `DFTMonitor` to benchmark 3D FIT against 3D-CWT across resolution levels and hole fill factors $f = \pi(r/a)^2 \in [4.5\%, 21.2\%]$.
4. **Identify the physical breakdown of single-mode CWT at large fill factors**: Show that when $r/a \gtrsim 0.16$ ($f \gtrsim 8\%$), the effective permittivity experienced by modes C/D in the PhC layer ($\varepsilon_0 - |\xi_{2,0}|$) drops below the cladding permittivity $\varepsilon_{\text{clad}}$, expelling the 3D electromagnetic mode vertically out of the etched PhC layer into the active core and causing single-mode perturbative CWT to overestimate radiation loss.
5. **Derive and design a double-lattice PCSEL for large-area single-mode operation**: Solve the 2D finite-size boundary-value problem $(\mathbf{K} + \mathbf{C})\mathbf{U} = (\delta + \mathrm{i}\alpha_{\text{th}}/2)\mathbf{U}$ for cavity sizes $L \in [50\;\mu\text{m}, 3000\;\mu\text{m}]$, analyze the unidirectional-coupling exceptional-point condition $C_{12} = 0$ ($R = 0, |I| = \mu$) alongside the double-lattice cancellation concept of Yoshida *et al.* (2019, 2023) [3, 4], and cross-check finite-cavity in-plane loss $\alpha_\parallel(L)$ against 2D effective-index FIT simulations.

---

## 1. Theory: 3D Coupled-Wave Theory and Double-Lattice Cancellation

### 1.1 Epitaxial Waveguide Stack and 2D Permittivity Fourier Expansion

Consider a square-lattice photonic-crystal surface-emitting laser with lattice constant $a = 290\;\text{nm}$ targeting $\lambda_0 \approx 964\;\text{nm}$ ($f_0 \approx 311\;\text{THz}$) [1–3]. Along the vertical growth axis $z$, the epitaxial heterostructure consists of:
- **Lower cladding** ($z < -150\;\text{nm}$): $\text{AlGaAs}$ with refractive index $n_{\text{clad}} = 3.26$ ($\varepsilon_{\text{clad}} = 10.6276$),
- **Active quantum-well (QW) core** ($z \in [-150, -30]\;\text{nm}$, thickness $t_{\text{act}} = 120\;\text{nm}$): $\text{InGaAs/GaAs}$ active layer with $n_{\text{act}} = 3.52$ ($\varepsilon_{\text{act}} = 12.3904$),
- **Photonic-crystal (PhC) layer** ($z \in [-30, +150]\;\text{nm}$, thickness $t_{\text{PhC}} = 180\;\text{nm}$): $\text{GaAs}$ host ($n_{\text{host}} = 3.46$, $\varepsilon_{\text{host}} = 11.9716$) perforated by circular air holes ($n_{\text{hole}} = 1.0$, $\varepsilon_{\text{hole}} = 1.0$) of radius $r$,
- **Upper cladding** ($z > +150\;\text{nm}$): $\text{AlGaAs}$ with $n_{\text{clad}} = 3.26$.

Inside the PhC region $z \in [z_{\text{PhC,min}}, z_{\text{PhC,max}}]$, the relative permittivity $\varepsilon(x, y, z)$ is periodic under $\mathbf{R} = n_x a\hat{\mathbf{x}} + n_y a\hat{\mathbf{y}}$ and expands in reciprocal-lattice vectors $\mathbf{G}_{m,n} = m\beta_0\hat{\mathbf{x}} + n\beta_0\hat{\mathbf{y}}$ with Bragg wavenumber $\beta_0 \equiv 2\pi/a$ (**derived here** following Liang *et al.* [1]):

$$
\varepsilon(x, y, z) = \varepsilon_0(z) + \Delta\varepsilon(x, y, z) = \varepsilon_0(z) + \Theta_{\text{PhC}}(z)\sum_{(m,n)\neq(0,0)} \xi_{m,n}\,\exp\!\left[-\mathrm{i}\beta_0(m x + n y)\right],
$$

where $\Theta_{\text{PhC}}(z) = 1$ inside the PhC layer and $0$ elsewhere. For a single circular hole of radius $r$ centered at $(0, 0)$ with area fill factor $f = \pi(r/a)^2$ and permittivity contrast $\Delta\varepsilon_{\text{PhC}} = \varepsilon_{\text{hole}} - \varepsilon_{\text{host}} < 0$:

$$
\varepsilon_0(z_{\text{PhC}}) = (1 - f)\varepsilon_{\text{host}} + f\varepsilon_{\text{hole}}, \qquad
\xi_{m,n}^{(1)} = \Delta\varepsilon_{\text{PhC}}\,\frac{2 f\,J_1\!\left(2\pi\sqrt{m^2+n^2}\,r/a\right)}{2\pi\sqrt{m^2+n^2}\,r/a}.
$$

For a **double-lattice unit cell** [3, 4] containing a primary hole of radius $r_1$ at $(0, 0)$ and a secondary hole of radius $r_2$ shifted by $(d_x, d_y)$, linear superposition of the non-overlapping apertures gives (**derived here**):

$$
\xi_{m,n}^{\text{double}} = \xi_{m,n}(r_1) + \xi_{m,n}(r_2)\,\exp\!\left[\mathrm{i}\,2\pi\left(m\frac{d_x}{a} + n\frac{d_y}{a}\right)\right].
$$

When the in-plane electric field $\mathbf{E}_\parallel$ has a component normal to the air-hole boundary, vectorial CWT [1] expands the inverse permittivity $\varepsilon^{-1}(x, y) = \zeta_0 + \sum_{(m,n)\neq(0,0)} \zeta_{m,n} \exp[-\mathrm{i}\beta_0(mx+ny)]$ with $\zeta_0 = (1-f)/\varepsilon_{\text{host}} + f/\varepsilon_{\text{hole}}$ and $\zeta_{m,n} = -\xi_{m,n}/(\varepsilon_{\text{host}}\varepsilon_{\text{hole}})$. Note that `wakis.pcsel_cwt` (`include_longitudinal=True`) does **not** evaluate Liang *et al.*'s $\zeta_{m,n}$ form: instead, it evaluates **an ad hoc local depolarisation term $\propto \xi\xi/\varepsilon_{\text{avg}}$** where $\varepsilon_{\text{avg}} = \varepsilon_0(z_{\text{PhC}})$ ($1/\varepsilon_{\text{avg}} = 0.0921 \neq \zeta_0 = 0.1768$ at $r/a = 0.18$, and $\varepsilon_{\text{avg}}^2\zeta\zeta$ would evaluate to $8.92\times$ the coded term).

### 1.2 Vertical Slab Eigenmode and 1D Green's Functions ($\Phi_{\text{rad}}$, $\Phi_{m,n}$)

In the unperturbed laterally averaged waveguide $\varepsilon_0(z)$, the fundamental $\text{TE}_0$ mode has vertical profile $\Theta_0(z)$ and effective index $n_{\text{eff}} = \beta_{\text{guided}}/k_0$ solving the 1D Helmholtz eigenvalue problem at the second-order Bragg wavenumber $\beta_{\text{guided}} = \beta_0 = 2\pi/a$ (**derived here** following [1]):

$$
\left[\frac{\mathrm{d}^2}{\mathrm{d}z^2} + k_B^2\varepsilon_0(z) - \beta_0^2\right]\Theta_0(z) = 0, \qquad \int_{-\infty}^{+\infty} |\Theta_0(z)|^2\,\mathrm{d}z = 1, \qquad k_B = \frac{\beta_0}{n_{\text{eff}}} = \frac{2\pi}{\lambda_{\text{Bragg}}}.
$$

At the $\Gamma$-point, the four basic guided waves propagate along $+\hat{\mathbf{x}}, -\hat{\mathbf{x}}, +\hat{\mathbf{y}}, -\hat{\mathbf{y}}$ with slowly varying envelopes $\mathbf{U}(x, y) = [R_x(x, y), S_x(x, y), R_y(x, y), S_y(x, y)]^T$:
- $R_x e^{-\mathrm{i}\beta_0 x}$ and $S_x e^{+\mathrm{i}\beta_0 x}$ carry TE electric field polarized along $\hat{\mathbf{y}}$,
- $R_y e^{-\mathrm{i}\beta_0 y}$ and $S_y e^{+\mathrm{i}\beta_0 y}$ carry TE electric field polarized along $\hat{\mathbf{x}}$.

First-order Bragg diffraction ($\Delta m = \pm 1$ or $\Delta n = \pm 1$) scatters the basic waves into the **$(0, 0)$ surface-normal radiation wave** ($k_x = k_y = 0$), governed by the 1D inhomogeneous vertical wave equation with outgoing Sommerfeld radiation boundary conditions $\partial_z \Phi_{\text{rad}} \mp \mathrm{i} k_B n_{\text{clad}}\Phi_{\text{rad}} = 0$ at $z \to \mp\infty$ (**derived here** following [1]):

$$
\left[\frac{\mathrm{d}^2}{\mathrm{d}z^2} + k_B^2\varepsilon_0(z)\right]\Phi_{\text{rad}}(z) = -\Theta_{\text{PhC}}(z)\Theta_0(z), \qquad
\mathcal{I}_{\text{rad}} \equiv \int_{z_{\text{PhC,min}}}^{z_{\text{PhC,max}}} \Theta_0(z)\Phi_{\text{rad}}(z)\,\mathrm{d}z.
$$

Multiplying the differential equation by $\Phi_{\text{rad}}^*(z)$, integrating over $z \in [z_{\min}, z_{\max}]$, and taking the imaginary part yields the **exact optical theorem / vertical power-conservation identity** (**derived here** and verified to $1.24\times 10^{-14}$ relative error in `wakis.pcsel_cwt`):

$$
-\operatorname{Im}(\mathcal{I}_{\text{rad}}) = k_B n_{\text{clad}}\left(|\Phi_{\text{rad}}(z_{\min})|^2 + |\Phi_{\text{rad}}(z_{\max})|^2\right) > 0.
$$

Similarly, higher-order Fourier harmonics $(m, n)$ with $m^2 + n^2 > 1$ lie outside the air/cladding light cone ($\beta_{m,n} = \beta_0\sqrt{m^2+n^2} > k_B n_{\text{clad}}$) and excite evanescently confined higher-order vertical waves governed by the 1D Green's function $\Phi_{m,n}(z, z')$ with decaying boundary conditions $\partial_z \Phi_{m,n} \pm \sqrt{\beta_{m,n}^2 - k_B^2\varepsilon_{\text{clad}}}\,\Phi_{m,n} = 0$:

$$
\left[\frac{\mathrm{d}^2}{\mathrm{d}z^2} + k_B^2\varepsilon_0(z) - \beta_0^2(m^2+n^2)\right]\Phi_{m,n}(z, z') = -\delta(z - z'), \qquad
\mathcal{I}_{m,n} = \iint_{\text{PhC}} \Theta_0(z)\Phi_{m,n}(z, z')\Theta_0(z')\,\mathrm{d}z\,\mathrm{d}z' > 0.
$$

### 1.3 The $4\times 4$ 3D-CWT Eigenproblem and $\Gamma$-Point Band-Edge Modes A–D

Projecting the 3D curl-curl Maxwell equation onto the four basic waves $(R_x, S_x, R_y, S_y)$ and eliminating the $(0, 0)$ radiative wave and $(m, n)$ higher-order waves yields the Noda group's 3D-CWT system (Liang *et al.*, *Phys. Rev. B* **84**, 195119, 2011 [1]; *Opt. Express* **20**, 15945, 2012 [2]):

$$
\begin{pmatrix}
-\mathrm{i}\partial_x & 0 & 0 & 0 \\
0 & +\mathrm{i}\partial_x & 0 & 0 \\
0 & 0 & -\mathrm{i}\partial_y & 0 \\
0 & 0 & 0 & +\mathrm{i}\partial_y
\end{pmatrix}
\begin{pmatrix} R_x \\ S_x \\ R_y \\ S_y \end{pmatrix}
+ \mathbf{C}
\begin{pmatrix} R_x \\ S_x \\ R_y \\ S_y \end{pmatrix}
= \left(\delta + \mathrm{i}\frac{\alpha}{2}\right)
\begin{pmatrix} R_x \\ S_x \\ R_y \\ S_y \end{pmatrix},
$$

where $\delta = n_{\text{eff}}(\omega - \omega_B)/c$ is the wavenumber detuning from the Bragg frequency $\omega_B = c\beta_0/n_{\text{eff}}$ and $\alpha = 2\operatorname{Im}\Lambda \ge 0$ is the modal power loss (or threshold power gain $\alpha_{\text{th}}$). In the convention implemented in `wakis.pcsel_cwt`, the $4\times 4$ coupling matrix $\mathbf{C} = \mathbf{C}_{1\text{D}} + \mathbf{C}_{\text{rad}} + \mathbf{C}_{2\text{D}}$ decomposes into three physical mechanisms (**derived here** following [1]):

1. **Direct 1D Bragg diffraction** ($\mathbf{C}_{1\text{D}}$) between counter-propagating waves ($R_x \leftrightarrow S_x$ via $\xi_{\pm 2,0}$ and $R_y \leftrightarrow S_y$ via $\xi_{0,\pm 2}$) weighted by the vertical PhC confinement factor $\Gamma_{\text{PhC}} = \int_{\text{PhC}} |\Theta_0(z)|^2\,\mathrm{d}z$:
   $$
   [\mathbf{C}_{1\text{D}}]_{12} = \kappa_{1\text{D}}^+ = -\frac{k_B^2}{2\beta_0}\xi_{2,0}\Gamma_{\text{PhC}}, \qquad
   [\mathbf{C}_{1\text{D}}]_{21} = \kappa_{1\text{D}}^- = -\frac{k_B^2}{2\beta_0}\xi_{-2,0}\Gamma_{\text{PhC}}.
   $$
   Because $\xi_{2,0} < 0$ for low-index air holes etched into a high-index semiconductor host, $\kappa_{1\text{D}} > 0$ ($\kappa_{1\text{D}} = +1454.68\;\text{cm}^{-1}$ at $r/a = 0.18$), placing the antisymmetric dark modes A and B (which have nodes on the air holes) on the low-frequency dielectric band ($\delta < 0$) and the symmetric modes C and D (which have antinodes on the air holes) on the high-frequency air band ($\delta > 0$). Direct first-order 2D coupling between orthogonal waves ($R_x, S_x$ polarized along $\hat{\mathbf{y}}$ and $R_y, S_y$ polarized along $\hat{\mathbf{x}}$) vanishes identically because $\hat{\mathbf{x}}\cdot\hat{\mathbf{y}} = 0$ [1].
2. **Radiative $(0, 0)$ wave coupling** ($\mathbf{C}_{\text{rad}}$), where the real part shifts the band-edge frequency and the positive-definite imaginary part $\mu > 0$ gives the vertical surface-emission loss constant [1]:
   $$
   [\mathbf{C}_{\text{rad}}]_{\alpha\beta} = -s_\alpha s_\beta\,\frac{k_B^2}{2\beta_0}\,\xi_{m_\alpha, n_\alpha}\xi_{-m_\beta, -n_\beta}\,\mathcal{I}_{\text{rad}}, \qquad
   \mu = -\frac{k_B^2}{2\beta_0}\operatorname{Im}\!\left(\xi_{1,0}\xi_{-1,0}\mathcal{I}_{\text{rad}}\right) > 0,
   $$
   with polarization-projection signs $s_\alpha \in \{+1, +1, -1, -1\}$ in the $(R_x, S_x, R_y, S_y)$ basis.
3. **Higher-order $(m, n)$ coupling** ($\mathbf{C}_{2\text{D}} = \mathbf{C}_{2\text{D}}^{(T)} + \mathbf{C}_{2\text{D}}^{(L)}$), which mediates both 1D corrections ($\eta_{1\text{D}}$) and the 2D in-plane coupling ($\kappa_{2\text{D}}$) between $x$- and $y$-propagating waves via intermediate harmonics $(m, n)$ with $m^2 + n^2 > 1$:
   - **Transverse TE Green's-function channel ($\mathbf{C}_{2\text{D}}^{(T)}$)** [1]: Driven by the transverse unit vector $\hat{\mathbf{t}}_{m,n} = (-n\hat{\mathbf{x}} + m\hat{\mathbf{y}})/\sqrt{m^2+n^2}$ and permittivity harmonics $\xi_{m,n}$ through the vertical Green's function overlap $\mathcal{I}_{m,n}$:
     $$
     [\mathbf{C}_{2\text{D}}^{(T)}]_{\alpha\beta} = -\frac{k_B^4}{2\beta_0}\sum_{m^2+n^2>1} (\hat{\mathbf{e}}_\alpha \cdot \hat{\mathbf{t}}_{m,n})(\hat{\mathbf{t}}_{m,n} \cdot \hat{\mathbf{e}}_\beta)\,\xi_{m_\alpha - m, n_\alpha - n}\,\xi_{m - m_\beta, n - n_\beta}\,\mathcal{I}_{m,n}.
     $$
     For 2D orthogonal coupling ($\alpha \in \{R_x, S_x\}$ with $\hat{\mathbf{e}}_\alpha = \hat{\mathbf{y}}$ and $\beta \in \{R_y, S_y\}$ with $\hat{\mathbf{e}}_\beta = \hat{\mathbf{x}}$), $(\hat{\mathbf{y}}\cdot\hat{\mathbf{t}}_{m,n})(\hat{\mathbf{t}}_{m,n}\cdot\hat{\mathbf{x}}) = -m n/(m^2+n^2)$, yielding $\kappa_{2\text{D}}^{(T)} = +40.89\;\text{cm}^{-1}$ ($\Delta f_{AB}^{(T)} = 0.235\;\text{THz}$) at $r/a = 0.18$.
   - **Ad hoc local depolarisation term $\propto \xi\xi/\varepsilon_{\text{avg}}$ ($\mathbf{C}_{2\text{D}}^{(L)}$, `include_longitudinal=True`; not Liang *et al.*'s $\zeta_{m,n}$ form)**: When `include_longitudinal=True`, `wakis.pcsel_cwt` (`scale_long = pref / stack.eps_phc_avg`) evaluates an ad hoc local depolarisation term along the longitudinal unit vector $\hat{\mathbf{l}}_{m,n} = (m\hat{\mathbf{x}} + n\hat{\mathbf{y}})/\sqrt{m^2+n^2}$ proportional to $\xi\xi/\varepsilon_{\text{avg}}$:
     $$
     [\mathbf{C}_{2\text{D}}^{(L)}]_{\alpha\beta} = \frac{k_B^2\Gamma_{\text{PhC}}}{2\beta_0\varepsilon_{\text{avg}}}\sum_{m^2+n^2>1} (\hat{\mathbf{e}}_\alpha \cdot \hat{\mathbf{l}}_{m,n})(\hat{\mathbf{l}}_{m,n} \cdot \hat{\mathbf{e}}_\beta)\,\xi_{m_\alpha - m, n_\alpha - n}\,\xi_{m - m_\beta, n - n_\beta},
     $$
     where $\varepsilon_{\text{avg}} = \varepsilon_0(z_{\text{PhC}}) = 10.855$ at $r/a = 0.18$ ($1/\varepsilon_{\text{avg}} = 0.0921 \neq \zeta_0 = 0.1768$, whereas $\varepsilon_{\text{avg}}^2\zeta\zeta$ evaluates to $8.92\times$ this coded term). At $r/a = 0.18$, this ad hoc term adds $\kappa_{2\text{D}}^{(L)} = +144.50\;\text{cm}^{-1}$, raising the total 2D coupling to $\kappa_{2\text{D}}^{(T+L)} = +185.39\;\text{cm}^{-1}$ ($\Delta f_{AB}^{(T+L)} = 1.065\;\text{THz}$ at $m_{\max}=3$, and $\kappa_{2\text{D}}^{(T+L)} = +202.03\;\text{cm}^{-1}$, $\Delta f_{AB}^{(T+L)} = 1.160\;\text{THz}$ at $m_{\max}=5$); with this ad hoc term the splitting gap drops from $6.82\times$ to $1.50\times$, and the form is not derived from [1].

In `wakis.pcsel_cwt`, we define the total 1D counter-propagating coupling parameters $R, I, \mu$ directly from the off-diagonal entries $C_{12}, C_{21}$ of the complete matrix $\mathbf{C} = \mathbf{C}_{1\text{D}} + \mathbf{C}_{\text{rad}} + \mathbf{C}_{2\text{D}}$:

$$
R \equiv \frac{1}{2}\operatorname{Re}(C_{12} + C_{21}), \qquad
I \equiv \frac{1}{2}\operatorname{Im}(C_{12} - C_{21}), \qquad
\mu \equiv \frac{1}{2}\operatorname{Im}(C_{12} + C_{21}),
$$

so that **in both the equations below and the code**:

$$
C_{12} = R + \mathrm{i}(\mu + I), \qquad
C_{21} = R + \mathrm{i}(\mu - I),
$$

where $R = \kappa_{1\text{D}} + \operatorname{Re}[\mathbf{C}_{\text{rad}}]_{12} + \operatorname{Re}[\mathbf{C}_{2\text{D}}]_{12}$ **already contains** the direct 1D Bragg coupling $\kappa_{1\text{D}}$. At $r/a = 0.18$ (transverse baseline), $\kappa_{1\text{D}} = +1454.68\;\text{cm}^{-1}$, $R = +1545.99\;\text{cm}^{-1}$, $I = 0$, $\mu = +103.09\;\text{cm}^{-1}$, and $|C_{12}| = |C_{21}| = \sqrt{R^2 + \mu^2} = 1549.42\;\text{cm}^{-1}$. For an inversion-symmetric single circular hole ($\xi_{m,n} = \xi_{-m,-n} \in \mathbb{R}$, $I = 0$), the $4\times 4$ $\Gamma$-point operator ($\partial_x = \partial_y = 0$) is:

$$
\mathbf{C}_{\text{single}} =
\begin{pmatrix}
\Delta_0 + \mathrm{i}\mu & R + \mathrm{i}\mu & \kappa_{2\text{D}} & -\kappa_{2\text{D}} \\
R + \mathrm{i}\mu & \Delta_0 + \mathrm{i}\mu & -\kappa_{2\text{D}} & \kappa_{2\text{D}} \\
\kappa_{2\text{D}} & -\kappa_{2\text{D}} & \Delta_0 + \mathrm{i}\mu & R + \mathrm{i}\mu \\
-\kappa_{2\text{D}} & \kappa_{2\text{D}} & R + \mathrm{i}\mu & \Delta_0 + \mathrm{i}\mu
\end{pmatrix}.
$$

Because $\mathbf{C}_{\text{single}}$ commutes with the $C_{4v}$ symmetry operations, its four exact eigenvectors and complex eigenvalues $\Lambda_\nu = \delta_\nu + \mathrm{i}\alpha_{v,\nu}/2$ are (**derived here**):
- **Mode A ($A_2$ irrep, antisymmetric dark BIC)**: $\mathbf{v}_A = \tfrac{1}{2}(1, -1, -1, 1)^T$,
  $$\delta_A = \Delta_0 - R - 2\kappa_{2\text{D}}, \qquad \alpha_{v,A} = 0 \quad (Q_A = \infty).$$
- **Mode B ($B_2$ irrep, quadrupolar dark BIC)**: $\mathbf{v}_B = \tfrac{1}{2}(1, -1, 1, -1)^T$,
  $$\delta_B = \Delta_0 - R + 2\kappa_{2\text{D}}, \qquad \alpha_{v,B} = 0 \quad (Q_B = \infty),$$
  split from Mode A by $\Delta\delta_{AB} = \delta_B - \delta_A = 4\kappa_{2\text{D}}$ ($\Delta f_{AB}^{(T)} = 0.235\;\text{THz}$ in transverse-only CWT, and $\Delta f_{AB}^{(T+L)} = 1.065\;\text{THz}$ when the ad hoc local depolarisation term $\propto \xi\xi/\varepsilon_{\text{avg}}$—not Liang *et al.*'s $\zeta_{m,n}$ form—is included at $r/a = 0.18$).
- **Modes C & D ($E$ irrep, degenerate dipole-active bright modes)**: $\mathbf{v}_C = \tfrac{1}{\sqrt{2}}(1, 1, 0, 0)^T$ and $\mathbf{v}_D = \tfrac{1}{\sqrt{2}}(0, 0, 1, 1)^T$,
  $$\delta_C = \delta_D = \Delta_0 + R, \qquad \alpha_{v,C} = \alpha_{v,D} = 4\mu > 0, \qquad Q_{C,D} = \frac{\beta_0}{\alpha_{v,C}} = \frac{\beta_0}{4\mu}.$$

Physically, in Modes A and B the counter-propagating wave pairs satisfy $R_x + S_x = 0$ and $R_y + S_y = 0$: their electric fields have a nodal line directly through the hole center, so the first-order $(0, 0)$ vertical radiation dipole moment cancels identically (**symmetry-protected bound states in the continuum, BICs**). In Modes C and D, $R_x = +S_x$ or $R_y = +S_y$ forms an antinode across the hole and radiates strongly into the surface normal with radiation constant $\alpha_v = 4\mu$.

### 1.4 Finite-Size Cavity Scaling and Double-Lattice Cancellation Condition

In a finite square PCSEL cavity of side length $L$ ($x, y \in [-L/2, +L/2]$) with no external reflection at the perimeter, the incoming wave envelopes must vanish at the four boundaries [2, 3]:

$$
R_x(-L/2, y) = 0, \qquad S_x(+L/2, y) = 0, \qquad R_y(x, -L/2) = 0, \qquad S_y(x, +L/2) = 0.
$$

Multiplying the finite-size 3D-CWT equation by $\mathbf{U}^\dagger$ and applying the divergence theorem over $[-L/2, +L/2]^2$ yields the **exact modal power balance identity** (**derived here**, verified to $< 1.5\times 10^{-10}$ relative error across all $L$ in `wakis.pcsel_cwt`):

$$
\alpha_{\text{th}} = \alpha_v + \alpha_\parallel, \qquad
\alpha_v = \frac{2\operatorname{Re}\iint_\Omega \mathbf{U}^\dagger\mathbf{C}_{\text{rad}}^{\text{skew}}\mathbf{U}\,\mathrm{d}x\,\mathrm{d}y}{\iint_\Omega |\mathbf{U}|^2\,\mathrm{d}x\,\mathrm{d}y}, \qquad
\alpha_\parallel = \frac{\oint_{\partial\Omega} |\mathbf{U}_{\text{out}}|^2\,\mathrm{d}\ell}{\iint_\Omega |\mathbf{U}|^2\,\mathrm{d}x\,\mathrm{d}y},
$$

where $\mathbf{C}_{\text{rad}}^{\text{skew}} \equiv (\mathbf{C}_{\text{rad}} - \mathbf{C}_{\text{rad}}^\dagger)/(2\mathrm{i})$ is the skew-Hermitian radiative operator, $\alpha_v$ is the vertical surface-radiation loss (which becomes strictly positive even for modes $A_1, B_1$ because the finite envelope breaks translational parity at the edges), and $\alpha_\parallel$ is the in-plane edge leakage loss.

**Derivation of the finite-size threshold-gap scaling law $\Delta\alpha_{\text{th}}(L)$ (derived here):**
In a single lattice ($r/a = 0.18$), the strong 1D backward coupling $|C_{12}| = \sqrt{R^2 + \mu^2} = 1549.4\;\text{cm}^{-1}$ implies $|C_{12}| L \gg 1$ for $L \ge 100\;\mu\text{m}$. In this over-coupled regime, the $(p, q)$ lateral envelope mode is quantized at wavenumber $\mathbf{k}_{p,q} \approx (p\pi/L, q\pi/L)$ with vanishing amplitude at $x, y = \pm L/2$:
1. **In-plane perimeter leakage ($\alpha_\parallel \propto L^{-3}$)**: Near $x, y = \pm L/2$, strong 1D Bragg reflection suppresses the outgoing boundary amplitude as $|\mathbf{U}_{\text{out}}(\pm L/2)| \sim (k_{p,q}/|C_{12}|)\|\mathbf{U}\|/L^{1/2}$, so the perimeter integral in Eq. (182) scales as $\alpha_\parallel^{(p,q)}(L) = c_\parallel^{(p,q)} L^{-3}$ with $c_\parallel^{(p,q)} \propto \pi^2(p^2+q^2)/|C_{12}|^2$. In `data/pcsel_summary.json`, the measured local log-slopes $\mathrm{d}\ln\alpha_{\parallel,A_1}/\mathrm{d}\ln L$ across the six consecutive intervals of $L \in [50, 100, 200, 500, 1000, 2000, 3000]\;\mu\text{m}$ are **$-2.86, -2.96, -2.99, -3.00, -3.00, -3.00 \to L^{-3}$**.
2. **Finite-envelope vertical radiation ($\alpha_v - \alpha_{v,0} \propto L^{-2}$)**: Although Modes A and B are dark BICs at $\mathbf{k}_\parallel = \mathbf{0}$, the finite-envelope gradient $-\mathrm{i}\nabla_\parallel \sim \mathbf{k}_{p,q}$ mixes the dark modes ($A, B$) at first order $\mathcal{O}(k_{p,q}/|C_{12}|)$ with the bright radiative doublet ($C, D$, which radiate at rate $4\mu$). By second-order non-Hermitian perturbation theory, the finite-size vertical radiation loss scales quadratically with $k_{p,q} \propto L^{-1}$:
   $$
   \alpha_v^{(p,q)}(L) = \alpha_{v,0} + c_v^{(p,q)} L^{-2} + \mathcal{O}(L^{-4}), \qquad c_v^{(p,q)} \propto \frac{\mu\,\pi^2(p^2+q^2)}{|C_{12}|^2}.
   $$
   In `data/pcsel_summary.json`, the measured local log-slopes $\mathrm{d}\ln\alpha_{v,A_1}/\mathrm{d}\ln L$ for the circular single lattice ($\alpha_{v,0} = 0$) are **$-1.83, -1.95, -1.99, -2.00, -2.00, -2.00 \to L^{-2}$**.
3. **Threshold gain margin ($\Delta\alpha_{\text{th}}(L) = \Delta c_v L^{-2} + \Delta c_\parallel L^{-3} \propto L^{-2}$ asymptotically)**: Taking the difference between the first competing mode and the fundamental mode yields the two-term asymptotic law (**derived here**):
   $$
   \Delta\alpha_{\text{th}}(L) \equiv \alpha_{\text{th}}^{(2)}(L) - \alpha_{\text{th}}^{(1)}(L) = \Delta c_v\,L^{-2} + \Delta c_\parallel\,L^{-3}, \qquad \Delta c_v \equiv c_v^{(2)} - c_v^{(1)} > 0, \quad \Delta c_\parallel \equiv c_\parallel^{(2)} - c_\parallel^{(1)} > 0.
   $$
   At intermediate sizes ($L \sim 100\text{--}500\;\mu\text{m}$), the $\Delta c_\parallel L^{-3}$ edge-leakage gap and $\Delta c_v L^{-2}$ vertical-radiation gap both contribute, steepening the effective log-slope past $-2$; for $L \gtrsim 1\;\text{mm}$, $\alpha_\parallel \ll \alpha_v$ and $\Delta\alpha_{\text{th}}(L)$ is governed by $\Delta c_v L^{-2} \propto L^{-2}$. In `data/pcsel_summary.json`, the measured local log-slopes $\mathrm{d}\ln(\Delta\alpha_{\text{th}})/\mathrm{d}\ln L$ across $[50\text{--}100, 100\text{--}200, 200\text{--}500, 500\text{--}1000, 1000\text{--}2000, 2000\text{--}3000]\;\mu\text{m}$ are **$-1.79, -2.48, -2.36, -2.21, -2.12, -2.07$** for the circular single lattice (and **$-0.75, -0.56, -1.88, -2.25, -2.14, -2.08$** for the $\alpha_v$-matched synthetic-phase single-lattice model), converging monotonically toward **$-2$ ($L^{-2}$)**. Importantly, **whether the relative threshold margin $\Delta\alpha_{\text{th}}/\alpha_{\text{th}}$ collapses at large $L$ depends on whether $\alpha_{v,0} > 0$ or $\alpha_{v,0} = 0$**: in our synthetic-phase single-lattice model (a synthetic Hermitian phase $\phi_{\text{Bragg}} = 0.4242\;\text{rad}$ applied directly to $\xi_{\pm 2,0}, \xi_{0,\pm 2}$ tuned to match $\alpha_{v,0} = 15.40\;\text{cm}^{-1}$, not a physical hole geometry, giving finite-cavity threshold $\alpha_{\text{th},A_1} = 15.61, 15.45, 15.40\;\text{cm}^{-1}$ at $L = 0.5, 1, 3\;\text{mm}$), $\Delta\alpha_{\text{th}}$ collapses from $1.834\;\text{cm}^{-1}$ ($\Delta\alpha_{\text{th}}/\alpha_{\text{th}} = 10.60\%$) at its optimum $L = 200\;\mu\text{m}$ to **$0.327\;\text{cm}^{-1}$ ($2.10\%$)** at $L = 500\;\mu\text{m}$, **$0.0688\;\text{cm}^{-1}$ ($0.445\%$)** at $L = 1\;\text{mm}$, and **$0.00674\;\text{cm}^{-1}$ ($0.0437\%$)** at $L = 3\;\text{mm}$. By contrast, in the **circular single lattice** ($\alpha_{v,0} = 0$), both $\Delta\alpha_{\text{th}} \propto L^{-2}$ and $\alpha_{\text{th},A_1} \propto L^{-2}$ vanish at the same asymptotic rate, so its relative margin stays at $\approx 11.1\%$ across every $L$ ($11.11\%$ at the $L = 3000\;\mu\text{m}$ sweep edge, which is the sweep boundary rather than an optimum).

**Analytical Derivation of the Unidirectional-Coupling Exceptional Point vs Double-Lattice Cancellation [3]:**
Consider a double-lattice unit cell with primary hole $r_1$ at $(0, 0)$ and secondary hole $r_2$ shifted diagonally by $(d, d)$ with relative hole area $\eta \approx (r_2/r_1)^2$. From Eq. (70), the Fourier harmonics are $\xi_{m,n}^{\text{double}} = \xi_{m,n}(r_1) + \xi_{m,n}(r_2)\exp[\mathrm{i}2\pi(m+n)d/a]$, and the 1D counter-propagating couplings in $\mathbf{C}$ are (**derived here**):

$$
C_{12} = R(d, r_2/r_1) + \mathrm{i}\!\left[\mu(d, r_2/r_1) + I(d, r_2/r_1)\right], \qquad
C_{21} = R(d, r_2/r_1) + \mathrm{i}\!\left[\mu(d, r_2/r_1) - I(d, r_2/r_1)\right].
$$

1. **Complete symmetric cancellation in Yoshida *et al.* (2019) [3]**: For two identical holes ($r_2 = r_1$, $\eta = 1$) separated by $d = a/4$, the first-order structure factor $1 + e^{\pm\mathrm{i}\pi/2}$ and second-order structure factor $1 + e^{\pm\mathrm{i}\pi} = 0$ interfere destructively so that $R, I,$ and $\mu$ all approach zero ($C_{12} = C_{21} \approx 0$). Because a practical laser requires finite surface emission ($\alpha_v > 0$) and weak residual feedback, Yoshida *et al.* [3, 4] detune $r_2/r_1 < 1$ and $d/a$ slightly above $0.25$.
2. **One-directional coupling / non-Hermitian exceptional point ($C_{12} = 0$, derived here)**: In our 3D-CWT formulation, we solve the exact coupled root system where the total Hermitian 1D feedback vanishes ($R = 0$) while the non-Hermitian radiative asymmetry balances the diagonal radiation damping ($|I| = \mu$, specifically $I = -\mu$):

$$
R(d^*, r_2^*/r_1) = 0, \qquad |I(d^*, r_2^*/r_1)| = \mu(d^*, r_2^*/r_1) \quad \Longrightarrow \quad C_{12} = 0, \quad |C_{21}| = 2\mu.
$$

Physically, $C_{12} = 0$ with $C_{21} \neq 0$ is a **non-Hermitian exceptional point (one-directional $180^\circ$ coupling)** of the $2\times 2$ 1D counter-propagating block: a wave traveling along $+x$ experiences zero direct 1D back-reflection into $-x$ ($C_{12} = 0$), so the geometric-mean 1D standing-wave feedback $\sqrt{C_{12}C_{21}} = 0$ vanishes identically while 2D orthogonal coupling ($\kappa_{2\text{D}}$) maintains four-wave coherence across the 2D aperture. Solving this system via continuous root refinement (`scipy.optimize.root` with self-consistent vertical slab and Green's functions at each $r_2/r_1$) yields **$d^*/a = 0.2768$** and **$(r_2/r_1)^* = 0.8882$**, where $|C_{12}| = 1.08\times 10^{-12}\;\text{cm}^{-1} \approx 0$, the remaining branch is reduced by $7.39\times$ ($|C_{21}| = 2\mu = 209.68\;\text{cm}^{-1}$ vs $1549.42\;\text{cm}^{-1}$ in the single lattice), and **our 3D-CWT infinite-lattice fundamental vertical radiation loss for this stack is $\alpha_{v,A}^\infty = 15.40\;\text{cm}^{-1}$**. At small cavities ($L \le 100\;\mu\text{m}$), the reduced feedback increases perimeter leakage so the double lattice has a smaller relative margin than the single lattice ($0.55\times$ at $L = 50\;\mu\text{m}$); above the first crossover length **$L_{\text{cross}} = 149.4\;\mu\text{m}$**, the double lattice increases the relative threshold margin $\Delta\alpha_{\text{th}}/\alpha_{\text{th}}$ over the $\alpha_v$-matched synthetic-phase single lattice by **$1.72\times$ at $L = 200\;\mu\text{m}$, $30.8\times$ at $L = 500\;\mu\text{m}$, $63.8\times$ at $L = 1\;\text{mm}$, and $95.6\times$ at $L = 3\;\text{mm}$** (and by **$6.09\times$** at $L = 500\;\mu\text{m}$ even against the synthetic-phase single lattice at its own optimum $L = 200\;\mu\text{m}$). However, because "single-lattice PCSEL suffers multimode threshold-margin collapse" is true only for absolute $\Delta\alpha_{\text{th}}$ or the $\alpha_v$-matched model, comparing against the **circular single lattice** (which keeps $\approx 11.1\%$ relative margin at every $L$) shows a **second crossover near $1.76\;\text{mm}$ ($1755.72\;\mu\text{m}$ by log interpolation)** beyond which the double-lattice relative margin ($8.93\%$ at $2\;\text{mm}$, $4.18\%$ at $3\;\text{mm}$) drops below the circular single lattice (`unperturbed_rel_margin_enhancement_ratio = 0.80` at $2\;\text{mm}$ and **$0.38$** at $3\;\text{mm}$).

In [ ]:
from __future__ import annotations

import argparse
import json
import os
import shlex
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import numpy as np

HERE = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

from _helpers import (  # noqa: E402
    DATA_DIR,
    DEFAULT_STACK,
    FIG_DIR,
    build_finite_pcsel_2d_solver,
    build_pcsel_unit_cell_solver,
    plot_all_figures,
    run_all_studies,
    run_finite_pcsel_2d_fit,
    run_pcsel_unit_cell,
    save_pcsel_setup_figures,
)
from wakis.pcsel_cwt import (  # noqa: E402
    compute_cwt_band_structure,
    compute_cwt_coupling_coefficients,
    double_lattice_coupling_scan,
    fourier_coefficients_phc,
    solve_finite_cwt,
    solve_infinite_cwt,
    solve_vertical_greens_functions,
    solve_vertical_slab_mode,
)

---

## 2. Simulation Setup & Mesh Schematics

Every Wakis 3D FIT simulation in this tutorial is preflighted with `wakis.estimate.estimate` and `preflight` and rendered with `wakis.schematic.save_setup_figures` before execution:
1. **3D FIT Bloch Unit-Cell Setup (`figures/00a_setup_geometry.png`, `figures/00b_setup_mesh_and_excitation.png`)**:
   - **Domain**: One square lattice period $L_x \times L_y = 290\;\text{nm} \times 290\;\text{nm}$ with Bloch-periodic (`Periodic` at $\mathbf{k}_\parallel = \mathbf{0}$ or `Bloch` at $\mathbf{k}_\parallel \neq \mathbf{0}$) lateral boundaries and 8-cell CPML absorbing boundaries on $\pm z$ ($z \in [-900, +900]\;\text{nm}$).
   - **Graded Mesh**: Fine uniform $\Delta x = \Delta y = 12.08\;\text{nm}$ ($N_x = N_y = 24$) and vertically graded $\Delta z_{\text{core}} = 12.0\;\text{nm}$ across the active QW and PhC layers ($z \in [-180, +180]\;\text{nm}$), expanding smoothly via `wakis.templates.meshing.graded_axis` to $\Delta z_{\text{clad}} = 35.0\;\text{nm}$ in the $\text{AlGaAs}$ claddings ($N_z = 73$, $42,048$ cells total). This provides **$22.7\;\text{cells}/\lambda$** in the highest-index active layer ($n_{\max} = 3.52$, exceeding the $\ge 20\;\text{cells}/\lambda$ rule) and **$8.64\;\text{cells}$ across the circular hole diameter** ($2r = 104.4\;\text{nm}$, exceeding the $\ge 4\;\text{cells}$ rule).
   - **Sources & Monitors**: Symmetry-projected `CurrentSource` pairs at $(\pm a/4, 0, z_{\text{act}})$ or $(0, \pm a/4, z_{\text{act}})$ selectively excite each $C_{4v}$ irreducible representation ($A_2$ Mode A, $B_2$ Mode B, and $E$ Modes C & D), recorded by an off-nodal `ResonanceMonitor` (`harminv`) and a 3D `DFTMonitor` slab across the PhC plane $z = +60\;\text{nm}$.

![3D FIT Unit-Cell Setup Geometry](figures/00a_setup_geometry.png)
![3D FIT Unit-Cell Mesh and Excitation](figures/00b_setup_mesh_and_excitation.png)

| Category | Component / Face | Specification |
| :--- | :--- | :--- |
| Domain | Size & bounds | Lx × Ly × Lz = 290 nm × 290 nm × 1.80 µm (z ∈ [-900 nm, 900 nm]) |
| Domain | Grid cells | Nx × Ny × Nz = 24 × 24 × 73 (42048 cells) |
| Discretization | Δx & resolution | Δ_min = 12 nm, 22.8 cells/λ (in n_max = 3.52), 8.7 cells across feature (104.4 nm) |
| Boundary | -x face | Periodic |
| Boundary | +x face | Periodic |
| Boundary | -y face | Periodic |
| Boundary | +y face | Periodic |
| Boundary | -z face | CPML (8 cells, 277 nm) |
| Boundary | +z face | CPML (8 cells, 277 nm) |
| Material | Background | Vacuum, ε = 1.00 (isotropic) |
| Material | Region 1: Active InGaAs/GaAs QW core (n=3.52) | ε_r = 12.39, μ_r = 1.00, model = Isotropic dielectric, x ∈ [-145 nm, 145 nm] (d = 290 nm) |
| Material | Region 2: PhC GaAs host layer (n=3.46) | ε_r = 11.97, μ_r = 1.00, model = Isotropic dielectric, x ∈ [-145 nm, 145 nm] (d = 290 nm) |
| Material | Region 3: Circular air hole (r/a=0.18, n=1.00) | ε_r = 1.00, μ_r = 1.00, model = Isotropic dielectric, x ∈ [-52.2 nm, 52.2 nm] (d = 104.4 nm) |
| Source | 1 · CurrentSource | pos: x = 72.5 nm, z = -60 nm, pol: J_y (E_y), pulse: f₀ = 311.0 THz, Δf = 18.00 THz |
| Source | 2 · CurrentSource | pos: x = -72.5 nm, z = -60 nm, pol: J_y (E_y), pulse: f₀ = 311.0 THz, Δf = 18.00 THz |
| Monitor | 3 · ResonanceMonitor | ResonanceMonitor (x = 72.5 nm), pos: x = 72.5 nm, freqs: time-domain |
| Monitor | 4 · DFTMonitor | DFTMonitor (z = 60 nm), pos: z = 60 nm, freqs: 3 freqs in [308.6, 313.2] THz |

2. **Finite-Cavity 2D Effective-Index FIT Setup (`figures/00a_setup_geometry_finite.png`, `figures/00b_setup_mesh_and_excitation_finite.png`)**:
   - **Domain & Resolution**: A finite $N_{\text{per}} \times N_{\text{per}}$ ($10\times 10$ shown, $L = 2.90\;\mu\text{m}$) square PCSEL array discretized at `cells_per_period = 22` ($\Delta x = \Delta y = 13.18\;\text{nm}$, providing **$21.4\;\text{cells}/\lambda \ge 20$** in $n_{\max} = 3.37$ and **$7.92\;\text{cells}$ across the circular hole diameter $\ge 4$**) with vertically projected effective permittivities $\varepsilon_{\text{host,2D}} = 11.35$, $\varepsilon_{\text{hole,2D}} = 8.33$ (calibrated to match the 3D-CWT effective index $n_{\text{eff}} = 3.3229$ and 1D coupling $|\kappa_{1\text{D}}|$) surrounded by 8-cell lateral CPML boundaries on $\pm x, \pm y$. **Excitation disclosure**: Rather than a single point dipole, the 2D FIT run uses a mode-matched $\cos(\pi x/L)\cos(\pi y/L)$ $A_2$ unit-cell `CurrentSource` array and corresponding symmetry-projected time monitor (`_helpers.py:build_finite_pcsel_2d_solver`) to project onto the fundamental $A_1$ envelope and suppress dense competing lateral/polarization modes within the $90\;\text{fs}$ ring-down window (it is a spatial symmetry/envelope projection, not post-hoc parameter tuning for monotonic $f$ and $Q$).

![Finite-Cavity 2D FIT Setup Geometry](figures/00a_setup_geometry_finite.png)
![Finite-Cavity 2D FIT Mesh and Excitation](figures/00b_setup_mesh_and_excitation_finite.png)

| Category | Component / Face | Specification |
| :--- | :--- | :--- |
| Domain | Size & bounds | Lx × Ly × Lz = 3.19 µm × 3.19 µm × 13.2 nm (z ∈ [-6.59 nm, 6.59 nm]) |
| Domain | Grid cells | Nx × Ny × Nz = 242 × 242 × 1 (58564 cells) |
| Discretization | Δx & resolution | Δ_min = 13.2 nm, 21.7 cells/λ (in n_max = 3.37), 7.9 cells across feature (104.4 nm) |
| Boundary | -x face | CPML (8 cells, 105.5 nm) |
| Boundary | +x face | CPML (8 cells, 105.5 nm) |
| Boundary | -y face | CPML (8 cells, 105.5 nm) |
| Boundary | +y face | CPML (8 cells, 105.5 nm) |
| Boundary | -z face | Periodic |
| Boundary | +z face | Periodic |
| Material | Background | Vacuum, ε = 1.00 (isotropic) |
| Material | Region 1: 10x10 Finite PCSEL Cavity (n_eff=3.323) | ε_r = 11.35, μ_r = 1.00, model = Isotropic dielectric, x ∈ [-1.45 µm, 1.45 µm] (d = 2.90 µm) |
| Material | Region 2: Etched PhC air hole array (r/a=0.18) | ε_r = 8.33, μ_r = 1.00, model = Isotropic dielectric, x ∈ [92.8 nm, 197.2 nm] (d = 104.4 nm) |
| Source | 1 · CurrentSource | pos: x = 217.5 nm, y = 145 nm, pol: J_y (E_y), pulse: f₀ = 308.7 THz, Δf = 35.00 THz |
| Monitor | 2 · ResonanceMonitor | ResonanceMonitor (x = 217.5 nm), pos: x = 217.5 nm, freqs: time-domain |
| Monitor | 3 · DFTMonitor | DFTMonitor (x ∈ [-1.45 µm, 1.45 µm]), pos: x ∈ [-1.45 µm, 1.45 µm], freqs: f = 300.0 THz |

In [ ]:
def load_or_run_summary(quick: bool = True) -> dict:
    """Load `data/pcsel_summary.json` if present, or execute the study."""
    json_path = DATA_DIR / "pcsel_summary.json"
    if json_path.exists():
        summary = json.loads(json_path.read_text(encoding="utf-8"))
    else:
        summary = run_all_studies(quick=quick)
        plot_all_figures()
    s1 = summary["slab_1d"]
    cwt = summary["cwt_infinite_r018"]
    fit = summary["fit_unit_cell_r018"]
    print(
        f"[1D Slab] n_eff = {s1['n_eff_fd']:.4f} (exact TMM {s1['n_eff_exact']:.4f}, "
        f"rel err = {s1['rel_err_n_eff']:.2e}), f_Bragg = {s1['f_bragg_thz']:.2f} THz, "
        f"Gamma_PhC = {s1['gamma_phc']*100:.2f}%, Gamma_act = {s1['gamma_act']*100:.2f}%"
    )
    print(
        f"[Gamma Modes r/a=0.18] FIT vs CWT: "
        f"f_A = {fit['freq_A_thz']:.3f} vs {cwt['freq_A_thz']:.3f} THz ({fit['rel_err_cwt_vs_fit_A_pct']:.3f}%), "
        f"f_B = {fit['freq_B_thz']:.3f} vs {cwt['freq_B_thz']:.3f} THz ({fit['rel_err_cwt_vs_fit_B_pct']:.3f}%), "
        f"f_CD = {fit['freq_CD_thz']:.3f} vs {cwt['freq_C_thz']:.3f} THz ({fit['rel_err_cwt_vs_fit_CD_pct']:.3f}%), "
        f"Q_CD = {fit['Q_CD']:.1f} vs {cwt['Q_C_dressed']:.1f} (dressed CWT)"
    )
    return summary

---

## 3. Results

### 3.1 Vertical Slab Waveguide Mode and 1D Green's Functions

Figure 1 (`figures/01_vertical_slab_and_greens_functions.png`) shows the 1D epitaxial permittivity profile $\varepsilon_0(z)$, the normalized fundamental $\text{TE}_0$ waveguide mode $\Theta_0(z)$, the complex $(0, 0)$ radiative Green's function $\Phi_{\text{rad}}(z)$, and the evanescently confined $(1, 1)$ and $(2, 0)$ Green's functions $\int \Phi_{m,n}(z, z')\Theta_{\text{PhC}}(z')\Theta_0(z')\,\mathrm{d}z'$ at $r/a = 0.18$ ($f = 10.18\%$):
- **Effective index & optical confinement**: The finite-difference 1D solver gives $n_{\text{eff}} = 3.3229$ ($f_{\text{Bragg}} = 311.10\;\text{THz}$), matching the exact root-finding transfer-matrix benchmark $n_{\text{eff}}^{\text{TMM}} = 3.3227$ to a relative error of $7.91\times 10^{-5}$. Because the etched air holes lower the average permittivity of the PhC layer to $\varepsilon_0(z_{\text{PhC}}) = 10.855 < \varepsilon_{\text{act}} = 12.390$, the $\text{TE}_0$ mode peaks inside the active QW layer ($\Gamma_{\text{act}} = 35.30\%$) while maintaining $\Gamma_{\text{PhC}} = 27.48\%$ overlap with the photonic-crystal grating.
- **Radiative vs evanescent Green's functions**: In Figure 1b, $\Phi_{\text{rad}}(z)$ oscillates sinusoidally with constant amplitude into both the upper and lower $\text{AlGaAs}$ claddings, carrying vertical surface emission and satisfying the optical theorem $-\operatorname{Im}(\mathcal{I}_{\text{rad}}) = 0.011705$ to $1.24\times 10^{-14}$ relative error. In contrast, the higher-order $(1, 1)$ and $(2, 0)$ Green's functions decay exponentially into the claddings ($\beta_{m,n} > k_B n_{\text{clad}}$), contributing purely reactive Hermitan coupling $\eta_{1\text{D}}$ without radiation loss.

![1D Vertical Slab Mode and Green's Functions](figures/01_vertical_slab_and_greens_functions.png)

---

### 3.2 3D FIT Unit-Cell $\Gamma$-Point Band-Edge Modes A–D and Field Symmetry

Figure 2 (`figures/02_gamma_band_edge_modes_3d_fit.png`) and the animated standing-wave oscillation (`figures/02_band_edge_modes_animation.gif`) show the four $\Gamma$-point band-edge modes extracted by `wakis.SolverFIT3D` at the fine production grid ($N_x\times N_y\times N_z = 24\times 24\times 73$, $\text{CPW} = 22.66$, $8.64\;\text{cells}$ across the hole diameter), with material contours (`overlay_geometry` circle of radius $r = 52.2\;\text{nm}$) and unit-cell boundary annotations:
- **Mode A ($A_2$, antisymmetric dark BIC)**: $f_A^{\text{FIT}} = 307.624\;\text{THz}$ ($f_A^{\text{Rich}} = 308.460\;\text{THz}$, $f_A^{\text{CWT}} = 308.639\;\text{THz}$, relative difference $0.330\%$ vs FIT and $0.058\%$ vs Richardson). The $E_y(x, y)$ pattern has a nodal line along $x = 0$ passing directly through the air hole center ($E_y(-x, y) = -E_y(x, y)$), and the in-plane vector field $(E_x, E_y)$ forms a closed azimuthal vortex around the hole, suppressing vertical radiation ($Q_A^{\text{FIT}} = 94098.1$, limited only by the $240\;\text{fs}$ finite ring-down window and discrete grid symmetry; $Q_A^{\text{CWT}} = \infty$).
- **Mode B ($B_2$, quadrupolar dark BIC)**: $f_B^{\text{FIT}} = 309.225\;\text{THz}$ ($f_B^{\text{Rich}} = 310.014\;\text{THz}$, transverse-only $f_B^{\text{CWT,(T)}} = 308.873\;\text{THz}$, $0.114\%$ vs FIT; with the ad hoc local depolarisation term $\propto \xi\xi/\varepsilon_{\text{avg}}$, $f_B^{\text{CWT,(T+L)}} = 309.704\;\text{THz}$, $0.155\%$ vs FIT). Its in-plane vector field forms a hyperbolic saddle (quadrupole) around the hole center, again yielding zero net dipole moment in the unit cell ($Q_B^{\text{FIT}} = 106919.3$, $Q_B^{\text{CWT}} = \infty$).
- **Modes C & D ($E$ doublet, bright surface-emitting modes)**: $f_C^{\text{FIT}} = 313.340\;\text{THz}$ and $f_D^{\text{FIT}} = 313.340\;\text{THz}$ ($f_{CD}^{\text{CWT,(T)}} = 313.196\;\text{THz}$, relative difference $0.046\%$ vs FIT and $0.324\%$ vs Richardson), with exact $C_{4v}$ degeneracy verified to $|f_C - f_D|/f_{CD} = 5.12\times 10^{-7}$. Because $E_y$ (Mode C) and $E_x$ (Mode D) are symmetric across the hole center ($E_y(-x, y) = +E_y(x, y)$), their antinodes overlap the low-index air hole, shifting them to the upper band edge and producing strong surface-normal radiation with $Q_C^{\text{FIT}} = 713.6$ and $Q_D^{\text{FIT}} = 713.3$ ($Q_{CD}^{\text{FIT}} = 713.5$ vs $Q_{CD}^{\text{CWT,dressed}} = 555.6$, $Q_{CD}^{\text{CWT,unpert}} = 525.4$).
- **Band gap and A–B splitting (transverse-only vs ad hoc local depolarisation term $\propto \xi\xi/\varepsilon_{\text{avg}}$)**: While individual absolute frequencies agree within $0.046\text{--}0.330\%$ of 3D FIT ($0.058\text{--}0.368\%$ vs Richardson, comparable to 3D FIT's $0.255\text{--}0.278\%$ fine-grid discretization error), our baseline transverse-only CWT (`include_longitudinal=False`, retaining only the $\hat{\mathbf{t}}_{m,n}$, $\xi_{m,n}$ channel in $\mathbf{C}_{2\text{D}}$) underpredicts the total band gap $f_{CD} - f_A$ by **$20.3\%$** ($4.557\;\text{THz}$ vs $5.716\;\text{THz}$ FIT) and underpredicts the A–B dark-mode splitting $\Delta f_{AB} = f_B - f_A$ by **$6.82\times$** ($\Delta f_{AB}^{\text{CWT,(T)}} = 0.235\;\text{THz}$, $\kappa_{2\text{D}}^{(T)} = 40.89\;\text{cm}^{-1}$ vs $\Delta f_{AB}^{\text{FIT}} = 1.601\;\text{THz}$, $1.554\;\text{THz}$ Richardson). Enabling `include_longitudinal=True` adds **an ad hoc local depolarisation term $\propto \xi\xi/\varepsilon_{\text{avg}}$ (not Liang *et al.*'s $\zeta_{m,n}$ form)** with $\kappa_{2\text{D}}^{(L)} = 144.50\;\text{cm}^{-1}$ ($\kappa_{2\text{D}}^{(T+L)} = 185.39\;\text{cm}^{-1}$ at $m_{\max}=3$ and $202.03\;\text{cm}^{-1}$ at $m_{\max}=5$), raising the CWT A–B splitting to **$\Delta f_{AB}^{\text{CWT,(T+L)}} = 1.065\;\text{THz}$** at $m_{\max}=3$ (**$1.160\;\text{THz}$** at $m_{\max}=5$) and bringing the total band gap $f_{CD} - f_A$ to **$5.989\;\text{THz}$** ($4.78\%$ vs 3D FIT $5.716\;\text{THz}$): **with this ad hoc term the splitting gap drops from $6.82\times$ to $1.50\times$ ($1.38\times$ at $m_{\max}=5$); the form is not derived from [1]**. Attributing the remaining $1.38\times\text{--}1.50\times$ difference to local boundary-field reconstruction (Li's inverse-rule coupling across the circular interface) and 3D TM-like ($E_z$) admixing in the asymmetric vertical stack is an untested hypothesis.

![3D FIT Gamma Band-Edge Modes A-D](figures/02_gamma_band_edge_modes_3d_fit.png)
![Animated Gamma Band-Edge Modes Oscillation](figures/02_band_edge_modes_animation.gif)

---

### 3.3 3D FIT Grid Resolution Convergence Ladder

Figure 3 (`figures/03_unit_cell_grid_convergence.png`) presents the 3-level 3D FIT grid convergence study across `coarse` ($16\times 16\times 62$, $\Delta x = 18.12\;\text{nm}$, $\text{CPW} = 15.11$, $5.76\;\text{cells}$/hole), `medium` ($20\times 20\times 67$, $\Delta x = 14.50\;\text{nm}$, $\text{CPW} = 18.89$, $7.20\;\text{cells}$/hole), and `fine` ($24\times 24\times 73$, $\Delta x = 12.08\;\text{nm}$, $\text{CPW} = 22.66$, $8.64\;\text{cells}$/hole):
- **Monotonic second-order convergence**: All four band-edge frequencies converge monotonically toward their Richardson extrapolations ($f_A^{\text{Rich}} = 308.460\;\text{THz}$, $f_B^{\text{Rich}} = 310.014\;\text{THz}$, $f_C^{\text{Rich}} = 314.214\;\text{THz}$, $f_D^{\text{Rich}} = 314.214\;\text{THz}$), with relative error against the Richardson asymptote decreasing from $0.595\text{--}0.643\%$ (`coarse`) to $0.367\text{--}0.400\%$ (`medium`) and $0.255\text{--}0.278\%$ (`fine`), following the $\mathcal{O}(\Delta x^2)$ slope guide.
- **Radiative quality factor convergence**: The leaky-mode radiative quality factor $Q_{CD}$ converges monotonically from $727.5$ (`coarse`, $2.54\%$ error vs $Q_{CD}^{\text{Rich}} = 709.5$) to $715.2$ (`medium`, $0.81\%$ error) and $713.5$ (`fine`, $0.56\%$ error).

![3D FIT Unit-Cell Grid Convergence](figures/03_unit_cell_grid_convergence.png)

---

### 3.4 Near-$\Gamma$ Band Structure ($\Gamma\text{--}X$, $\Gamma\text{--}M$) and Off-$\Gamma$ Radiative $Q(\mathbf{k}_\parallel)$

Figure 4 (`figures/04_band_structure_and_radiative_q.png`) compares the 3D-CWT dispersion $\omega(\mathbf{k}_\parallel)$ against 3D Bloch-periodic Wakis FIT simulations (`solver.bloch_k = (k_x, k_y, 0)` on the fine $N_x = 24$, $\text{CPW} = 22.66$ mesh with Bloch phase compensation and physical positive-$Q$ pole selection) along $\Gamma\text{--}X$ ($\mathbf{k}_\parallel = (k, 0)$) and $\Gamma\text{--}M$ ($\mathbf{k}_\parallel = (k, k)/\sqrt{2}$) for $|k|a/(2\pi) \le 0.045$:
- **Band curvature and mode hybridization**: At $\mathbf{k}_\parallel = \mathbf{0}$, Modes A and B sit at the lower band edges ($308.64\;\text{THz}$ and $308.87\;\text{THz}$ in transverse-only CWT, $309.70\;\text{THz}$ with the ad hoc local depolarisation term $\propto \xi\xi/\varepsilon_{\text{avg}}$; $307.62\;\text{THz}$ and $309.22\;\text{THz}$ in FIT) while the degenerate doublet C/D sits at the upper band edge ($313.20\;\text{THz}$ in transverse-only CWT, $314.63\;\text{THz}$ with the ad hoc local depolarisation term $\propto \xi\xi/\varepsilon_{\text{avg}}$; $313.34\;\text{THz}$ in FIT). Moving away from $\Gamma$, the linear derivative operator $\mathbf{K}(\mathbf{k}_\parallel) = \operatorname{diag}(k_x, -k_x, k_y, -k_y)$ splits the four branches into two steep air-band / dielectric-band branches ($\partial\omega/\partial k \approx \pm c/n_{\text{eff}}$) and two nearly flat transverse branches.
- **Symmetry-breaking radiation of BICs off $\Gamma$ & non-monotonic Band-2 $Q(k_x)$**: In Figure 4b, Modes A and B have infinite radiative $Q$ at $\mathbf{k}_\parallel = \mathbf{0}$ ($Q_A, Q_B \propto |\mathbf{k}_\parallel|^{-2} \to \infty$ as $\mathbf{k}_\parallel \to \mathbf{0}$) because $C_2$ rotational symmetry forbids coupling to the normal-incidence plane wave. Any non-zero in-plane wavevector $\mathbf{k}_\parallel \neq \mathbf{0}$ (such as the fundamental envelope wavenumber $k_\parallel \approx \pi/L$ of a finite $L\times L$ cavity) mixes Modes A and B with the bright doublet C and D, opening a finite radiative channel $\alpha_v(k_\parallel) \propto (k_\parallel / |C_{12}|)^2\mu$ that provides the surface-normal output beam of a finite PCSEL. Note that along $\Gamma\text{--}X$, Band 2 (ordered by ascending frequency $f_1 < f_2 < f_3 < f_4$ in `bands_fit`) exhibits a non-monotonic quality factor in both 3D FIT ($Q_2^{\text{FIT}} = 106919 \to 1861.4 \to 2704.1 \to 1785.6$ at $k_x a/(2\pi) = 0, 0.015, 0.030, 0.045$) and 3D-CWT (Figure 4b, green curve, which dips to $Q_2^{\text{CWT}} = 6998$ near $k_{\text{red}} \approx 0.004$ before rebounding to $61568$ at $k_{\text{red}} = 0.030$): at $k_x = 0$, Mode B is the equal-weight quadrupolar state $\frac{1}{2}(1, -1, 1, -1)^T$; for small $k_x > 0$ below the A–B hybridization crossover ($c k_x / n_{\text{eff}} \sim \Delta\omega_{AB}$), the $\pm k_x$ Doppler shift on $(R_x, S_x)$ strongly admixes the bright $x$-dipole $(R_x + S_x)$, lowering $Q_2$; once $c k_x / n_{\text{eff}}$ exceeds $\Delta\omega_{AB}$ ($k_{\text{red}} \gtrsim 0.02$ in FIT where $\Delta f_{AB}^{\text{FIT}} = 1.60\;\text{THz}$), the $x$-propagating waves decouple into the steep dispersive branches and Band 2 rotates toward the purely $y$-propagating dark standing wave $\frac{1}{\sqrt{2}}(0, 0, 1, -1)^T$ ($R_y = -S_y$), whose first-order $(0, 0)$ radiative overlap vanishes along $\Gamma\text{--}X$, causing $Q_2(k_x)$ to rebound upward at $k_{\text{red}} = 0.030$.

![Near-Gamma Band Structure and Radiative Q](figures/04_band_structure_and_radiative_q.png)

---

### 3.5 Hole Radius / Fill-Factor Sweep and Where Single-Mode 3D-CWT Breaks Down

Figure 5 (`figures/05_hole_radius_sweep_and_cwt_breakdown.png`) benchmarks 3D-CWT against 3D Wakis FIT on the fine mesh ($N_x = N_y = 24, \Delta z_{\text{core}} = 12\;\text{nm}$, $\text{CPW} = 22.66 \ge 20$, $5.76\text{--}12.48\;\text{cells}$ across the hole diameter) across hole radii $r/a \in \{0.12, 0.15, 0.18, 0.22, 0.26\}$ (fill factors $f = \pi(r/a)^2 \in \{4.52\%, 7.07\%, 10.18\%, 15.21\%, 21.24\%\}$):
- **Band-edge frequencies and ad hoc local depolarisation term $\propto \xi\xi/\varepsilon_{\text{avg}}$ splitting (Figure 5a)**: Across $f \in [4.5\%, 21.2\%]$, 3D-CWT tracks all four 3D FIT band-edge frequencies within **$0.017\%\text{--}0.347\%$** (transverse baseline), reproducing the upward blue-shift as low-index air replaces $\text{GaAs}$ ($n_{\text{eff}}$ drops from $3.376$ at $r/a = 0.08$ to $3.277$ at $r/a = 0.28$). As shown by the two green curves in Figure 5a, transverse-only CWT (`include_longitudinal=False`, dotted green) predicts $\Delta f_{AB}^{(T)} \in [0.056, 0.562]\;\text{THz}$ across $r/a \in [0.12, 0.26]$ vs $\Delta f_{AB}^{\text{FIT}} \in [0.422, 2.498]\;\text{THz}$ in 3D FIT, whereas including the ad hoc local depolarisation term $\propto \xi\xi/\varepsilon_{\text{avg}}$ (`include_longitudinal=True`, dashed dark green; not Liang *et al.*'s $\zeta_{m,n}$ form) gives $\Delta f_{AB}^{(T+L)} \in [0.185, 2.371]\;\text{THz}$ in `data/pcsel_summary.json` across $r/a \in [0.12, 0.26]$ (deviating from 3D FIT $\Delta f_{AB}^{\text{FIT}}$ by $-56.1\%$ at $r/a = 0.12$, $-33.5\%$ at $r/a = 0.18$, and $-5.1\%$ at $r/a = 0.26$; note that an unscaled $[0.224, 3.095]\;\text{THz}$ range would deviate by $+24\%$ at the largest $r/a = 0.26$, $3.095\;\text{THz}$ vs $2.498\;\text{THz}$, rather than closely tracking across the entire sweep).
- **Radiative $Q_{CD}$ and the large-fill-factor breakdown of single-mode CWT (Figures 5b–c)**:
  - At small to moderate fill factors ($r/a \le 0.15$, $f \le 7.1\%$), 3D-CWT and 3D FIT agree closely on the leaky-mode quality factor: at $r/a = 0.15$ ($f = 7.07\%$), $f_{CD}^{\text{CWT}} = 311.687\;\text{THz}$ vs $f_{CD}^{\text{FIT}} = 312.103\;\text{THz}$ ($0.133\%$ error) and $Q_{CD}^{\text{FIT}} = 1062.3$ vs $Q_{CD}^{\text{CWT,dressed}} = 1026.4$ (**$3.38\%$ relative error**).
  - However, for $r/a \ge 0.22$ ($f \ge 15.2\%$), 3D FIT shows $Q_{CD}^{\text{FIT}}$ **rebounding upward** from $644.9$ ($r/a = 0.22$) to $1274.8$ ($r/a = 0.26$, $f_{CD}^{\text{FIT}} = 315.479\;\text{THz}$ vs $f_{CD}^{\text{CWT}} = 315.815\;\text{THz}$), whereas single-mode perturbative 3D-CWT predicts a monotonic decrease ($Q_{CD}^{\text{CWT}} = 295.0 \to 186.7$, a $54.3\% \to 85.4\%$ discrepancy).
  - **Physical explanation (Figure 5c, derived here)**: In Modes C and D, the in-plane standing wave $\cos(\beta_0 x)$ peaks inside the low-index air holes, so the *modal* effective permittivity inside the PhC layer is $\varepsilon_{\text{PhC,eff}}^{(C,D)} = \varepsilon_0(z_{\text{PhC}}) - |\xi_{2,0}|$ rather than the spatial average $\varepsilon_0(z_{\text{PhC}})$. As shown in Figure 5c, $\varepsilon_{\text{PhC,eff}}^{(C,D)}$ drops below the cladding permittivity $\varepsilon_{\text{clad}} = 10.63$ at $r/a \approx 0.16$ ($f \approx 8\%$), and even the average permittivity $\varepsilon_0(z_{\text{PhC}})$ drops below $\varepsilon_{\text{clad}}$ at $r/a \approx 0.20$ ($f \approx 12.6\%$). Once $\varepsilon_{\text{PhC,eff}}^{(C,D)} < \varepsilon_{\text{clad}}$, the PhC layer acts as a low-index barrier for Modes C and D, expelling the true 3D wave vertically into the high-index active layer ($n_{\text{act}} = 3.52$) and sharply reducing the actual field overlap with the holes in 3D FIT. Single-mode perturbative CWT uses a single common vertical profile $\Theta_0(z)$ for all four modes and becomes inaccurate for $Q_{CD}$ once $f \gtrsim 12\%$.

![Hole Radius Sweep and CWT Breakdown](figures/05_hole_radius_sweep_and_cwt_breakdown.png)

---

### 3.6 Double-Lattice PCSEL: Self-Consistent Scan and Exceptional-Point Root $(R = 0, |I| = \mu \implies C_{12} = 0)$

Figure 6 (`figures/06_double_lattice_coupling_cancellation.png`) maps the 1D feedback behavior across hole-pair separation $d/a \in [0.23, 0.30]$ and secondary hole radius ratio $r_2/r_1 \in [0.65, 0.96]$ for a diagonal double lattice $(d_x, d_y) = (d, d)$ with primary hole $r_1/a = 0.18$, recomputing the 1D vertical slab mode $\Theta_0(z)$ and vertical Green's functions $\Phi_{\text{rad}}(z), \Phi_{m,n}(z, z')$ self-consistently at each $r_2/r_1$:
- **Single-lattice baseline**: In the circular single lattice ($r_2 = 0$), the 1D backward coupling is $|C_{12}| = |C_{21}| = \sqrt{R^2 + \mu^2} = 1549.4\;\text{cm}^{-1}$ ($R = +1546.0\;\text{cm}^{-1}$, containing $\kappa_{1\text{D}} = +1454.7\;\text{cm}^{-1}$; $I = 0$; $\mu = 103.1\;\text{cm}^{-1}$).
- **Unidirectional-coupling exceptional-point root ($C_{12} = 0$)**: Refining the grid minimum via `scipy.optimize.root` on the coupled system $(R(d, r_2/r_1) = 0,\; |I(d, r_2/r_1)| - \mu(d, r_2/r_1) = 0)$ locates the exact continuous root at **$d^*/a = 0.2768$** and **$(r_2/r_1)^* = 0.8882$**. At this root, $R = -1.04\times 10^{-12}\;\text{cm}^{-1}$ and $I = -104.84\;\text{cm}^{-1} = -\mu$, so $C_{12} = R + \mathrm{i}(\mu + I)$ vanishes to machine precision (**$\min(|C_{12}|, |C_{21}|) = 1.08\times 10^{-12}\;\text{cm}^{-1}$**), while the opposite branch is reduced by **$7.39\times$** to $|C_{21}| = 2\mu = 209.7\;\text{cm}^{-1}$ and **our 3D-CWT infinite-lattice fundamental mode vertical radiation loss for this stack settles at $\alpha_{v,A}^\infty = 15.40\;\text{cm}^{-1}$**.

![Double-Lattice Coupling Cancellation](figures/06_double_lattice_coupling_cancellation.png)

---

### 3.7 Finite-Size PCSEL Threshold Gain $\alpha_{\text{th}}(L)$ and Fair Single-Lattice Baselines

Figure 7 (`figures/07_finite_size_pcsel_threshold_and_double_lattice.png`) compares the finite-size 3D-CWT threshold gain $\alpha_{\text{th}}(L) = \alpha_v(L) + \alpha_\parallel(L)$ and the **relative threshold gain margin** $\Delta\alpha_{\text{th}}(L)/\alpha_{\text{th},A_1}(L)$ across cavity sizes $L \in \{50, 100, 200, 500, 1000, 2000, 3000\}\;\mu\text{m}$ for the circular single lattice ($\alpha_{v,0} = 0$, where `single_circ_sweep_edge_L_um = 3000` is the sweep edge, not an optimum), the **$\alpha_v$-matched synthetic-phase single-lattice model** (a synthetic phase `bragg_phase_rad = 0.4242 rad` on $\xi_{\pm 2,0}, \xi_{0,\pm 2}$ in $\mathbf{C}_{1\text{D}}$ tuned to match $\alpha_{v,0} = 15.40\;\text{cm}^{-1}$, not a physical hole geometry), and the continuous-root double lattice ($d^*/a = 0.2768, (r_2/r_1)^* = 0.8882$, $\alpha_{v,0} = 15.40\;\text{cm}^{-1}$), alongside the resolved 2D effective-index FIT cross-check (`finite_2d_fit`):
- **Single-lattice scaling $\Delta\alpha_{\text{th}}(L) = \Delta c_v L^{-2} + \Delta c_\parallel L^{-3} \to L^{-2}$ (Figures 7a–b)**: In the circular single lattice, strong in-plane trapping ($|C_{12}| = 1549.4\;\text{cm}^{-1}$) causes $\alpha_\parallel \propto L^{-3}$ (measured local log-slopes $-2.86 \to -3.00$) and $\alpha_v \propto L^{-2}$ (measured local log-slopes $-1.83 \to -2.00$), so the absolute threshold margin $\Delta\alpha_{\text{th}}$ decays with local log-slopes **$-1.79, -2.48, -2.36, -2.21, -2.12, -2.07$** across $L \in [50, 3000]\;\mu\text{m}$ from $4.103\;\text{cm}^{-1}$ ($L = 50\;\mu\text{m}$) and $1.189\;\text{cm}^{-1}$ ($L = 100\;\mu\text{m}$) to **$0.00531\;\text{cm}^{-1}$ ($L = 1\;\text{mm}$)** and **$0.000528\;\text{cm}^{-1}$ ($L = 3\;\text{mm}$)**; however, because its fundamental threshold $\alpha_{\text{th},A_1}(L)$ also decays as $L^{-2}$ ($\alpha_{v,0} = 0$), the **circular single lattice keeps $\approx 11.1\%$ relative margin at every $L$** ($11.10\%$ at $1\;\text{mm}$, $11.11\%$ at $2\;\text{mm}$ and at the $3\;\text{mm}$ sweep edge). Thus, **"single-lattice PCSEL suffers multimode threshold-margin collapse" is true only for absolute $\Delta\alpha_{\text{th}}$ or the $\alpha_v$-matched synthetic-phase model**: in the synthetic-phase single-lattice model (where $\phi_{\text{Bragg}} = 0.4242\;\text{rad}$ on $\xi_{\pm 2,0}, \xi_{0,\pm 2}$ yields $\alpha_{v,0} = 15.40\;\text{cm}^{-1}$ and finite-cavity threshold $\alpha_{\text{th},A_1} = 17.30, 15.61, 15.45, 15.40\;\text{cm}^{-1}$ at $L = 0.2, 0.5, 1, 3\;\text{mm}$), the relative margin reaches its maximum at $L_{\text{opt}}^{\text{single}} = 200\;\mu\text{m}$ ($\Delta\alpha_{\text{th}}/\alpha_{\text{th}} = 10.60\%$, $\Delta\alpha_{\text{th}} = 1.834\;\text{cm}^{-1}$) and then collapses with local log-slopes $-1.88 \to -2.25 \to -2.14 \to -2.08 \approx L^{-2}$ to **$2.10\%$ ($\Delta\alpha_{\text{th}} = 0.327\;\text{cm}^{-1}$) at $L = 500\;\mu\text{m}$**, **$0.445\%$ ($\Delta\alpha_{\text{th}} = 0.0688\;\text{cm}^{-1}$) at $L = 1\;\text{mm}$**, and **$0.0437\%$ ($\Delta\alpha_{\text{th}} = 0.00674\;\text{cm}^{-1}$) at $L = 3\;\text{mm}$**.
- **Small-cavity trade-off, first crossover at $L_{\text{cross}} = 149.4\;\mu\text{m}$, and second crossover near $1.76\;\text{mm}$ vs the circular single lattice (Figures 7a–b)**: At $L = 50\;\mu\text{m}$ and $L = 100\;\mu\text{m}$, the double lattice has a *smaller* relative threshold margin than the single lattice (**$3.00\%$ vs $5.49\%$ at $L = 50\;\mu\text{m}$, a $0.55\times$ ratio** vs the synthetic-phase single lattice and $0.50\times$ vs the circular single lattice; **$6.05\%$ vs $10.19\%$ at $L = 100\;\mu\text{m}$, $0.59\times$**) because reduced 1D feedback raises in-plane perimeter leakage ($\alpha_{\text{th},A_1} = 862.7\;\text{cm}^{-1}$ at $L = 50\;\mu\text{m}$). The double-lattice relative margin overtakes the synthetic-phase single lattice at **$L_{\text{cross}} = 149.4\;\mu\text{m}$** ($151.9\;\mu\text{m}$ vs the circular single lattice) and peaks at **$L = 500\;\mu\text{m}$ with $\Delta\alpha_{\text{th}}/\alpha_{\text{th}} = 64.55\%$** ($\alpha_{\text{th},A_1} = 34.48\;\text{cm}^{-1}$, $\Delta\alpha_{\text{th}} = 22.26\;\text{cm}^{-1}$)—a **$30.8\times$ enhancement** over the same-$L$ synthetic-phase single lattice ($2.10\%$) and a **$6.09\times$ enhancement** over the synthetic-phase single lattice at its own optimum ($10.60\%$ at $L = 200\;\mu\text{m}$). At millimeter scales, the double lattice maintains **$28.40\%$ at $L = 1\;\text{mm}$** ($\alpha_{\text{th},A_1} = 20.51\;\text{cm}^{-1}$, $\Delta\alpha_{\text{th}} = 5.83\;\text{cm}^{-1}$, **$63.8\times$** over the same-$L$ synthetic-phase single lattice and $2.56\times$ over the circular single lattice), then falls to **$8.93\%$ at $L = 2\;\text{mm}$** and **$4.18\%$ at $L = 3\;\text{mm}$** ($\alpha_{\text{th},A_1} = 16.00\;\text{cm}^{-1}$, $\Delta\alpha_{\text{th}} = 0.669\;\text{cm}^{-1}$, **$95.6\times$** over the synthetic-phase single lattice). Against the **circular single lattice** ($\approx 11.1\%$ at every $L$), however, the double lattice undergoes a **second crossover near $1.76\;\text{mm}$ ($1755.72\;\mu\text{m}$ by log interpolation)**, falling below the circular single lattice to **`unperturbed_rel_margin_enhancement_ratio = 0.80` ($8.93\%$ vs $11.11\%$) at $L = 2\;\text{mm}$ and `0.38` ($4.18\%$ vs $11.11\%$) at $L = 3\;\text{mm}$**.
- **Per-$L$ mode envelopes and small-cavity 2D FIT cross-check (Figure 7c)**: Figure 7c plots separate normalized intensity profiles $|\mathbf{U}(x, 0)|^2$ at $L = 200\;\mu\text{m}$ and $L = 500\;\mu\text{m}$ for both the single lattice and the double lattice, showing how the single-lattice profile remains pinched in the interior ($\cos^2(\pi x/L)$) at both $L = 200\;\mu\text{m}$ and $L = 500\;\mu\text{m}$ whereas the double-lattice fundamental mode expands across the aperture. In Figure 7a (upper-left markers), we also cross-check the in-plane edge leakage $\alpha_\parallel(L)$ against resolved 2D effective-index Wakis FIT simulations (`cells_per_period = 22`, $\text{CPW} = 21.43 \ge 20$, $7.92\;\text{cells}$/hole, excited by the mode-matched $\cos(\pi x/L)\cos(\pi y/L)$ $A_2$ unit-cell source array disclosed in Section 2) for small under-coupled cavities ($N_{\text{per}} \in \{8, 10, 12, 16\}$, $L \in [2.32, 4.64]\;\mu\text{m}$, $\kappa_{1\text{D}} L \in [0.34, 0.67] < 1$): as $N_{\text{per}}$ increases from $8$ to $16$, $f^{\text{FIT}}$ progresses from $295.35$ to $303.51\;\text{THz}$ and $Q_\parallel^{\text{FIT}}$ increases from $7.94$ to $20.60$, agreeing with 3D-CWT $\alpha_\parallel$ within $6.29\%$ at $N_{\text{per}} = 8$ ($\alpha_\parallel^{\text{FIT}} = 27275\;\text{cm}^{-1}$ vs $\alpha_\parallel^{\text{CWT}} = 25661\;\text{cm}^{-1}$) and $14.2\%\text{--}15.9\%$ at $N_{\text{per}} = 10\text{--}16$.

![Finite-Size PCSEL Threshold Scaling and Double Lattice](figures/07_finite_size_pcsel_threshold_and_double_lattice.png)

In [ ]:
def print_finite_cavity_scaling(summary: dict) -> None:
    """Print finite-size single-lattice vs double-lattice threshold margin table."""
    print(
        "L [um] | Synth-phase rel [%] | Circ single rel [%] | Double alpha_th [cm^-1] | Double rel [%] | D/S_synth | D/S_circ"
    )
    print("-" * 119)
    for row in summary["finite_cwt_sweep"]:
        print(
            f"{row['L_um']:6.0f} | "
            f"{row['single_matched_rel_margin_pct']:19.4f} | "
            f"{row['single_rel_margin_pct']:19.2f} | "
            f"{row['double_A1_alpha_th_cm']:23.2f} | "
            f"{row['double_rel_margin_pct']:14.2f} | "
            f"{row['matched_rel_margin_enhancement_ratio']:8.2f}x | "
            f"{row['unperturbed_rel_margin_enhancement_ratio']:7.2f}x"
        )

---

## Quantitative Summary

Every numerical value in the table below is read directly from `data/pcsel_summary.json` (generated by `pcsel.py --full`) and verified automatically in `tests/test_tut_07_10_pcsel.py`.

| Study / Physical Quantity | Analytical / 3D-CWT Result | Wakis 3D / 2D FIT Result | Relative Error / Comparison | Reference / Provenance |
| :--- | :--- | :--- | :--- | :--- |
| **1D Slab**: Fundamental $\text{TE}_0$ effective index $n_{\text{eff}}$ ($r/a = 0.18$) | $3.3227$ (exact TMM), $f_{\text{Bragg}} = 311.10\;\text{THz}$ | $3.3229$ (FD solver, $\Gamma_{\text{PhC}} = 27.48\%$, $\Gamma_{\text{act}} = 35.30\%$) | $7.91\times 10^{-5}$ | Slab formulation of [1] (this stack) |
| **1D Green's Function**: Optical theorem $-\operatorname{Im}(\mathcal{I}_{\text{rad}})$ | $0.011705$ (boundary flux RHS) | $0.011705$ (PhC overlap LHS) | $1.24\times 10^{-14}$ (machine precision) | Derived here from [1] |
| **Unit Cell ($r/a=0.18$)**: Mode A ($A_2$, dark BIC) frequency & $Q$ | $f_A^{\text{CWT}} = 308.639\;\text{THz}$, $Q_A^{\text{CWT}} = \infty$ | $f_A^{\text{FIT}} = 307.624\;\text{THz}$ ($f_A^{\text{Rich}} = 308.460\;\text{THz}$), $Q_A^{\text{FIT}} = 94098.1$ | $0.330\%$ vs FIT ($0.058\%$ vs Rich) | 3D-CWT [1] vs Wakis 3D FIT (this stack) |
| **Unit Cell ($r/a=0.18$)**: Mode B ($B_2$, dark BIC) & A–B split (trans. vs +ad hoc $\propto \xi\xi/\varepsilon_{\text{avg}}$) | $f_B^{(T)} = 308.873\;\text{THz}$ ($\Delta f_{AB}^{(T)} = 0.235\;\text{THz}$); $f_B^{(T+L)} = 309.704\;\text{THz}$ ($\Delta f_{AB}^{(T+L)} = 1.065\;\text{THz}$) | $f_B^{\text{FIT}} = 309.225\;\text{THz}$ ($f_B^{\text{Rich}} = 310.014\;\text{THz}$), $Q_B^{\text{FIT}} = 106919.3$, $\Delta f_{AB}^{\text{FIT}} = 1.601\;\text{THz}$ | $0.114\%$ ($T$) / $0.155\%$ ($T+L$) freq diff; $\Delta f_{AB}$ gap drops $6.82\times \to 1.50\times$ | 3D-CWT [1] + ad hoc $\propto \xi\xi/\varepsilon_{\text{avg}}$ (not from [1]) vs 3D FIT |
| **Unit Cell ($r/a=0.18$)**: Modes C & D ($E$ doublet) frequency & $C_{4v}$ split | $f_{CD}^{\text{CWT}} = 313.196\;\text{THz}$ ($\Delta f_{CD} = 0$) | $f_{CD}^{\text{FIT}} = 313.340\;\text{THz}$ ($f_C^{\text{Rich}} = 314.214\;\text{THz}$) | $0.046\%$ vs FIT, $|f_C-f_D|/f_{CD} = 5.12\times 10^{-7}$ | 3D-CWT [1] vs Wakis 3D FIT (this stack) |
| **Unit Cell ($r/a=0.18$)**: Leaky-mode radiative $Q_{CD}$ | $Q_{CD}^{\text{CWT,dressed}} = 555.6$ ($Q_{CD}^{\text{unpert}} = 525.4$) | $Q_{CD}^{\text{FIT}} = 713.5$ ($Q_{CD}^{\text{Rich}} = 709.5$) | $22.1\%$ ($0.56\%$ FIT fine vs Rich) | 3D-CWT [1] vs Wakis 3D FIT (this stack) |
| **3D FIT Convergence**: `coarse` $\to$ `medium` $\to$ `fine` ($f_A$ err vs Rich) | $\mathcal{O}(\Delta x^2)$ ($p_{\text{obs}} = 2.41$) | $0.595\% \to 0.391\% \to 0.271\%$ ($\text{CPW} = 15.11 \to 18.89 \to 22.66$) | Monotonic 3-level convergence | Wakis `SolverFIT3D` (this stack) |
| **Radius Sweep ($r/a=0.15$)**: Moderate fill factor ($f = 7.07\%$) $f$ & $Q_{CD}$ | $f_{CD}^{\text{CWT}} = 311.687\;\text{THz}$, $Q_{CD}^{\text{CWT}} = 1026.4$ | $f_{CD}^{\text{FIT}} = 312.103\;\text{THz}$, $Q_{CD}^{\text{FIT}} = 1062.3$ | $0.133\%$ freq err, $3.38\%$ $Q_{CD}$ err | 3D-CWT [1] vs Wakis 3D FIT (this stack) |
| **Radius Sweep ($r/a=0.26$)**: Large fill factor ($f = 21.24\%$) CWT breakdown | $f_{CD}^{\text{CWT}} = 315.815\;\text{THz}$, $Q_{CD}^{\text{CWT}} = 186.7$ | $f_{CD}^{\text{FIT}} = 315.479\;\text{THz}$, $Q_{CD}^{\text{FIT}} = 1274.8$ | $0.106\%$ freq err, $85.4\%$ $Q$ breakdown ($\varepsilon_{\text{PhC,eff}} < \varepsilon_{\text{clad}}$) | Derived here (this stack) |
| **Double Lattice ($d^*/a=0.2768, r_2/r_1=0.888$)**: Exceptional point $C_{12}=0$ & $\alpha_{v,A}^\infty$ | $\min(\|C_{12}\|,\|C_{21}\|) = 1.08\times 10^{-12}\;\text{cm}^{-1}$, $\alpha_{v,A}^\infty = 15.40\;\text{cm}^{-1}$ | $\max(\|C_{12}\|,\|C_{21}\|) = 209.7\;\text{cm}^{-1}$ vs $1549.4\;\text{cm}^{-1}$ (single) | One-directional 1D coupling ($7.39\times$ other branch reduction) | Derived here (our 3D-CWT root for this stack, cf. [3]) |
| **Finite PCSEL ($L=50\;\mu\text{m}$ & Crossover)**: Small-$L$ trade-off & $L_{\text{cross}}$ | Asym. single ($L=50\;\mu\text{m}$): $5.49\%$ ($\Delta\alpha_{\text{th}} = 4.56\;\text{cm}^{-1}$); $L_{\text{opt}}^{\text{single}} = 200\;\mu\text{m}$ ($10.60\%$) | Double ($L=50\;\mu\text{m}$): $3.00\%$ ($\Delta\alpha_{\text{th}} = 25.86\;\text{cm}^{-1}$); crossover $L_{\text{cross}} = 149.4\;\mu\text{m}$ | $0.55\times$ at $L = 50\;\mu\text{m}$ ($>1\times$ for $L > 149.4\;\mu\text{m}$) | Finite 3D-CWT derived here (this stack) |
| **Finite PCSEL ($L=500\;\mu\text{m}$)**: Relative margin $\Delta\alpha_{\text{th}}/\alpha_{\text{th}}$ (at $\alpha_{v,0}=15.40\;\text{cm}^{-1}$) | Asym. single: $2.10\%$ ($\Delta\alpha_{\text{th}} = 0.327\;\text{cm}^{-1}$) | Double: $64.6\%$ ($\alpha_{\text{th},A_1} = 34.48\;\text{cm}^{-1}$, $\Delta\alpha_{\text{th}} = 22.26\;\text{cm}^{-1}$) | $30.8\times$ vs same-$L$ single ($6.09\times$ vs single $L_{\text{opt}}=200\;\mu\text{m}$) | Finite 3D-CWT [2, 3] (this stack) |
| **Finite PCSEL ($L=1000\;\mu\text{m}$)**: $1\;\text{mm}$ relative margin $\Delta\alpha_{\text{th}}/\alpha_{\text{th}}$ | Asym. single: $0.445\%$ ($\Delta\alpha_{\text{th}} = 0.0688\;\text{cm}^{-1}$) | Double: $28.4\%$ ($\alpha_{\text{th},A_1} = 20.51\;\text{cm}^{-1}$, $\Delta\alpha_{\text{th}} = 5.83\;\text{cm}^{-1}$) | $63.8\times$ relative margin enhancement | Finite 3D-CWT [2, 3] (this stack) |
| **Finite PCSEL ($L=3000\;\mu\text{m}$)**: $3\;\text{mm}$ relative margin $\Delta\alpha_{\text{th}}/\alpha_{\text{th}}$ | Synth.-phase single: $0.0437\%$ ($\Delta\alpha_{\text{th}} = 0.00674\;\text{cm}^{-1}$); Circ. single: $11.11\%$ (sweep edge) | Double: $4.18\%$ ($\alpha_{\text{th},A_1} = 16.00\;\text{cm}^{-1}$, $\Delta\alpha_{\text{th}} = 0.669\;\text{cm}^{-1}$, 2nd crossover $1755.72\;\mu\text{m}$ / $1.76\;\text{mm}$) | $95.6\times$ vs $\alpha_v$-matched; $0.38\times$ vs circ. single ($0.80\times$ at $2\;\text{mm}$) | Finite 3D-CWT [2, 3] (this stack) |
| **Small Finite Cavity ($N_{\text{per}}=8, L=2.32\;\mu\text{m}$)**: Qualitative $\alpha_\parallel$ cross-check | $\alpha_\parallel^{\text{CWT}} = 25661\;\text{cm}^{-1}$ ($Q_\parallel^{\text{CWT}} = 8.44$) | $\alpha_\parallel^{\text{FIT}} = 27275\;\text{cm}^{-1}$ ($Q_\parallel^{\text{FIT}} = 7.94$, $\text{CPW} = 21.4$) | $6.29\%$ relative difference ($\kappa_{1\text{D}} L < 1$ regime) | 2D FIT vs 3D-CWT (this stack) |

---

## Limits & Failure Modes

1. **Underestimation of A–B Dark-Mode Splitting and Ad Hoc Local Depolarisation Term $\propto \xi\xi/\varepsilon_{\text{avg}}$ (Not Liang *et al.*'s $\zeta_{m,n}$ Form)**:
   - *Assumption*: Baseline transverse-only 3D-CWT (`include_longitudinal=False`) retains only the transverse TE unit vector $\hat{\mathbf{t}}_{m,n} = (-n\hat{\mathbf{x}} + m\hat{\mathbf{y}})/\sqrt{m^2+n^2}$ and direct permittivity harmonics $\xi_{m,n}$ in the higher-order coupling matrix $\mathbf{C}_{2\text{D}}$. When `include_longitudinal=True`, `wakis.pcsel_cwt.compute_cwt_coupling_coefficients` adds an ad hoc local depolarisation term $\propto \xi\xi/\varepsilon_{\text{avg}}$ (not Liang *et al.*'s $\zeta_{m,n}$ form, since $1/\varepsilon_{\text{avg}} = 0.0921 \neq \zeta_0 = 0.1768$ and $\varepsilon_{\text{avg}}^2\zeta\zeta$ evaluates to $8.92\times$ the coded term).
   - *Counter-case*: Transverse-only CWT underpredicts $\Delta f_{AB} = f_B - f_A$ by $6.82\times$ at $r/a = 0.18$ ($0.235\;\text{THz}$ vs $1.601\;\text{THz}$ in 3D FIT) and underpredicts the band gap $f_{CD} - f_A$ by $20.3\%$ ($4.557\;\text{THz}$ vs $5.716\;\text{THz}$). Enabling `include_longitudinal=True` raises $\kappa_{2\text{D}}$ from $40.89$ to $185.39\;\text{cm}^{-1}$ and $\Delta f_{AB}$ to $1.065\;\text{THz}$ ($1.160\;\text{THz}$ at $m_{\max}=5$): with this ad hoc term the splitting gap drops from $6.82\times$ to $1.50\times$ ($1.38\times$ at $m_{\max}=5$); the form is not derived from [1]. Attributing the residual $1.38\times\text{--}1.50\times$ difference to cross-boundary inverse-rule coupling (Li's rule) and 3D TM-like ($E_z$) admixing in the asymmetric vertical stack is an untested hypothesis.
2. **Large Fill-Factor Breakdown of Single-Mode Perturbative 3D-CWT ($f = \pi(r/a)^2 \gtrsim 12\%$, Figure 5b–c)**:
   - *Assumption*: Standard 3D-CWT [1, 2] expands all four basic guided waves around a **single** unperturbed vertical slab eigenmode $\Theta_0(z)$ computed from the laterally averaged permittivity $\varepsilon_0(z_{\text{PhC}}) = (1-f)\varepsilon_{\text{host}} + f\varepsilon_{\text{hole}}$.
   - *Counter-case*: Because the bright modes C and D peak inside the low-index air holes ($\cos(\beta_0 x)$ antinode at $x = 0$), their effective PhC permittivity $\varepsilon_{\text{PhC,eff}}^{(C,D)} = \varepsilon_0(z_{\text{PhC}}) - |\xi_{2,0}|$ drops below the cladding permittivity $\varepsilon_{\text{clad}} = 10.63$ for $r/a \gtrsim 0.16$ ($f \gtrsim 8\%$). In 3D FIT, this expels the vertical mode profile of Modes C and D out of the PhC layer into the underlying active layer ($n_{\text{act}} = 3.52$), causing $Q_{CD}^{\text{FIT}}$ to rebound upward from $644.9$ ($r/a = 0.22$) to $1274.8$ ($r/a = 0.26$), whereas single-mode CWT underpredicts $Q_{CD}$ by $54\%\text{--}85\%$. Accurate modeling for $f \gtrsim 15\%$ requires either full 3D FIT or a multi-vertical-mode CWT / Bloch-mode expansion.
3. **Slowly Varying Envelope Approximation (SVEA) at Small Cavities ($N_{\text{per}} \le 16$, $\kappa_{1\text{D}} L < 1$)**:
   - *Assumption*: Finite-size 3D-CWT assumes that the wave envelopes $R_{x,y}(x, y), S_{x,y}(x, y)$ vary slowly over a single lattice period $a$ ($|\partial_x R_x| \ll \beta_0 |R_x|$) and imposes zero incoming wave boundary conditions at $x, y = \pm L/2$, neglecting Fresnel impedance mismatch at the PhC/cladding mesa edge.
   - *Counter-case*: In the small-cavity 2D FIT cross-check ($N_{\text{per}} \in \{8, 10, 12, 16\}$, $L \in [2.32, 4.64]\;\mu\text{m}$), the normalized coupling strength is $\kappa_{1\text{D}} L \in [0.34, 0.67] < 1$ (deep in the under-coupled regime), so the standing-wave envelope changes substantially over a few periods and experiences weak facet reflection at the finite array boundary, resulting in a $6.3\%\text{--}15.9\%$ difference between 2D FIT and SVEA CWT $\alpha_\parallel(L)$ alongside a finite-size red-shift of the resonance frequency ($295.35 \to 303.51\;\text{THz}$).
4. **Cold-Cavity Threshold vs High-Power Continuous-Wave Carrier & Thermal Lensing**:
   - *Assumption*: The present tutorial solves the linear cold-cavity eigenvalue problem with spatially uniform refractive index $n(x, y) = \text{const}$ and uniform gain, without active semiconductor carrier-density rate equations or self-heating.
   - *Counter-case*: While double-lattice coupling cancellation provides a large cold-cavity threshold margin $\Delta\alpha_{\text{th}}/\alpha_{\text{th}}$ at $L = 1\text{--}3\;\text{mm}$ [3], achieving $50\;\text{W}$-class continuous-wave single-mode operation at $L = 3\;\text{mm}$ [4] cannot be explained by cold-cavity CWT alone: central temperature rise $\Delta T(x, y) \sim 10\text{--}30\;\text{K}$ creates a parabolic thermal index lens $\Delta n_{\text{therm}}(x, y) = (\partial n/\partial T)\Delta T(x, y) \sim +2\times 10^{-3}$ while spatial hole burning depletes carriers at the mode peak (increasing $n$ via the linewidth enhancement factor $\alpha_H$). If the thermal/carrier index step $\Delta n\,k_0$ exceeds the band-edge curvature dispersion, the fundamental mode self-focuses and narrows, requiring a pre-chirped lattice pitch $a(x, y)$ in addition to double-lattice cancellation [4].

---

## References

1. Y. Liang, C. Peng, K. Sakai, S. Iwahashi, and S. Noda, "Three-dimensional coupled-wave model for square-lattice photonic crystal lasers with transverse electric polarization: A general approach," *Phys. Rev. B* **84**(19), 195119 (2011), DOI: `10.1103/PhysRevB.84.195119`.
2. Y. Liang, C. Peng, K. Sakai, S. Iwahashi, and S. Noda, "Three-dimensional coupled-wave analysis for square-lattice photonic crystal surface emitting lasers with transverse-electric polarization: finite-size effects," *Opt. Express* **20**(14), 15945–15961 (2012), DOI: `10.1364/OE.20.015945`.
3. M. Yoshida, M. De Zoysa, K. Ishizaki, Y. Tanaka, M. Kawasaki, R. Hatsuda, B. Song, J. Gelleta, and S. Noda, "Double-lattice photonic-crystal resonators enabling high-brightness semiconductor lasers with a symmetric narrow-divergence beam," *Nature Materials* **18**(2), 121–128 (2019), DOI: `10.1038/s41563-018-0242-y`.
4. M. Yoshida, S. Katsuno, T. Inoue, J. Gelleta, K. Izumi, M. De Zoysa, K. Ishizaki, and S. Noda, "High-brightness scalable continuous-wave single-mode photonic-crystal laser," *Nature* **618**(7966), 727–732 (2023), DOI: `10.1038/s41586-023-06059-8`.
5. K. Sakai, E. Miyai, T. Sakaguchi, D. Ohnishi, T. Kitamura, and S. Noda, "Lasing band-edge identification for a surface-emitting photonic crystal laser," *IEEE J. Sel. Areas Commun.* **23**(7), 1335–1340 (2005), DOI: `10.1109/JSAC.2005.851205`.
6. C. W. Hsu, B. Zhen, A. D. Stone, J. D. Joannopoulos, and M. Soljačić, "Bound states in the continuum," *Nature Reviews Materials* **1**(9), 16048 (2016), DOI: `10.1038/natrevmats.2016.48`.

In [ ]:
def parse_args(argv=None):
    """Parse CLI arguments for Tutorial 7.10 (`--quick`, `--full`, `--plot`)."""
    ap = argparse.ArgumentParser(
        description="Tutorial 7.10 · Photonic-crystal surface-emitting laser (PCSEL): band-edge modes and 3D-CWT"
    )
    ap.add_argument(
        "--quick",
        action="store_true",
        help="Run fast verification study or load precomputed data and generate figures.",
    )
    ap.add_argument(
        "--full",
        action="store_true",
        help="Force full production 3D FIT convergence, band structure, radius sweep, and finite-size CWT runs.",
    )
    ap.add_argument(
        "--plot",
        action="store_true",
        help="Regenerate all figures in figures/ from existing data/.",
    )
    if argv is None:
        env_args = os.environ.get("WAKIS_TUTORIAL_ARGS")
        argv = shlex.split(env_args) if env_args else sys.argv[1:]
    args, _ = ap.parse_known_args(argv)
    return args


def main(argv=None):
    """CLI and notebook entrypoint for Tutorial 7.10 (`pcsel.py`)."""
    args = parse_args(argv)
    if args.plot:
        paths = plot_all_figures()
        print(f"Generated {len(paths)} figures in {FIG_DIR}.")
        return 0

    if args.full:
        summary = run_all_studies(quick=False)
        paths = plot_all_figures()
        print_finite_cavity_scaling(summary)
        print(f"Completed full PCSEL study; {len(paths)} figures saved in {FIG_DIR}.")
        return 0

    summary = load_or_run_summary(quick=True)
    if not (FIG_DIR / "07_finite_size_pcsel_threshold_and_double_lattice.png").exists():
        plot_all_figures()
    print_finite_cavity_scaling(summary)
    return 0


if __name__ == "__main__":
    main()